# **Proyecto Integrador**
# ***LUMINA* - La Biblia Viva con IA**

Plataforma conversacional que facilita la comprensión y aplicación contextual del texto bíblico mediante NLP y RAG, con un enfoque responsable, trazable y centrado en el usuario.

---

> *"La explicación de tus palabras nos da luz; y da entendimiento a los de mente sencilla."*
>
> — Salmos 119:130

---

### **Avance 4: Modelos alternativos**
**Equipo:** *#19*  
**Curso:** *Proyecto Integrador*  
**Posgrado:** *MNA - Maestría en Inteligencia Artificial Aplicada*  
**Institución:** Tecnológico de Monterrey  
**Fecha:** *22 de febrero del 2026*  

---

### **Miembros del equipo**
- **A01732505** - Steven Sebastian Brutscher Cortez
- **A01795323** - Anghelo Daniel Pérez Martínez
- **A01423059** - Esmeralda González García


---

> *"Me dediqué entonces a adquirir conocimiento, a explorar y a investigar para encontrar la sabiduría y la razón de ser de las cosas"*
>
> — Eclesiastés 7:25

---

# **Introducción**

El presente entregable corresponde al **Avance 4 del Proyecto Integrador LUMINA La Biblia Viva con IA**. Mientras que el Avance 3 se centró en construir un baseline funcional y reproducible para responder preguntas en lenguaje natural con respaldo bíblico, este avance da el siguiente paso metodológico: evaluar múltiples variaciones del sistema RAG con el fin de seleccionar la arquitectura más robusta y trazable para el caso de uso pastoral-cristiano.

A diferencia de problemas de ML supervisado (con variable objetivo etiquetada), el objetivo de LUMINA es recuperar evidencia bíblica relevante y producir respuestas fieles al contexto. Por ello, la evaluación debe considerar tres dimensiones complementarias: (i) retrieval (qué tan bien recuperamos pasajes pertinentes), (ii) groundedness (en qué medida las respuestas se anclan al contexto recuperado) y (iii) calidad comunicativa de la respuesta (relevancia, fidelidad, precisión de citas y tono pastoral).
Con base en los artefactos consolidados en el Avance 2 (chunking, metadatos tabulares y embeddings sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) y en el baseline del Avance 3 (implementado en Google Colab y ChromaDB para la recuperación), en este Avance 4 realizamos una evaluación comparativa de las siguientes variantes:

* LLM-only (sin contexto externo).
* Dense-RAG (similitud coseno sobre embeddings normalizados).
* Hybrid-RAG (combinación BM25 + Dense con normalización y mezcla lineal).
* Versiones “repaired” (re-escritura forzada para usar exclusivamente el contexto recuperado cuando se detecta fuga o citas fuera de contexto).

# **Objetivos de aprendizaje**

En concordancia con las instrucciones del curso y adaptados al contexto específico de *LUMINA*, los objetivos de este avance son:

- Evaluar el **desempeño** de modelos **RAG híbridos** y densos frente a un modelo **LLM-only** para determinar la reducción de alucinaciones.

- Implementar y medir el impacto de un proceso de **Repair Pass (reparación automática)** en la calidad y precisión de las respuestas.

- Utilizar métricas avanzadas como **GCF1** para distinguir no solo la calidad lingüística, sino el nivel de **groundedness (sustento en la evidencia)** de cada respuesta.

- Analizar la viabilidad técnica de los modelos propuestos considerando la latencia y la trazabilidad de las fuentes bíblicas.

# **Relación con el Avance 3 – Baseline**

El Avance 3 permitió para *LUMINA* establecer una línea base sólida donde el uso de **Dense-RAG** e **Hybrid-RAG** demostró alcanzar un desempeño máximo en el juez con cero fuga de información. Este avance toma esos resultados para refinar la selección del modelo, confirmando que, bajo la métrica GCF1, el modelo Dense-RAG muestra una ligera superioridad en el uso del contexto frente a la versión híbrida.

# **Conexión con CRISP-ML(Q)**

- Dentro de CRISP-ML(Q), esta fase se sitúa en la etapa 4 de **Modelado y Evaluación (Modeling and Evaluation)**. Aquí se realiza el ajuste fino de la arquitectura del modelo y se valida rigurosamente su robustez antes de pasar al despliegue, asegurando que el sistema sea capaz de detectar y corregir bajas puntuaciones de fidelidad mediante mecanismos condicionados de reparación.



---

> *"Si alguno de ustedes quiere construir una torre, ¿no se sienta primero a calcular los gastos y ver si tiene lo suficiente para terminarla?"*
>
> — Lucas 14:28

---

In [ ]:
# ============================================================
# Montando Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# Inicializando las librerías necesarias
# ============================================================

import os
import json
import re
from datetime import datetime

import numpy as np
import pandas as pd

---

> *"Y me buscaréis y hallaréis, porque me buscaréis de todo vuestro corazón."*
>
> — Jeremías 29:13

---

# **Parte 1 — Configuración del entorno y carga de artefactos (Avance 2)**

En esta primera parte preparamos el entorno para construir el **Baseline** de LUMINA. En particular:

1. **Definimos rutas** de trabajo (notebook actual y carpetas del Avance 2).
2. **Cargamos los artefactos** generados previamente:
   - Dataset tabular de chunks (`.parquet`) con texto y metadatos canónicos.
   - Embeddings semánticos (`.npz`) para RV1909 y VBL.
   - Manifest (`.json`) para trazabilidad.
3. Realizamos **sanity checks** críticos:
   - Verificar shapes y consistencia (número de chunks).
   - Verificar alineación por `chunk_id` entre tabular y embeddings.
   - Revisar columnas disponibles y ejemplos.

**Meta de esta sección:** garantizar que estamos construyendo el baseline sobre datos consistentes y reproducibles, antes de indexar y evaluar retrieval.


In [ ]:
# ============================================================
# Parte 1 — Configuración del entorno y carga de artefactos (A2)
# ============================================================

# -----------------------------
# 1) Rutas (Drive / Colab)
# -----------------------------
# Notebook actual (Avance 4)
A4_DIR = "/content/drive/MyDrive/Proyecto Integrador/data"

# Carpeta del Avance 2 donde quedaron artefactos y embeddings
A2_DIR = "/content/drive/MyDrive/Proyecto Integrador/"

A2_ARTIFACTS_DIR = os.path.join(A2_DIR, "artifacts")
A2_EMB_DIR = os.path.join(A2_DIR, "embeddings")

print("A4_DIR:", A4_DIR)
print("A2_DIR:", A2_DIR)
print("A2_ARTIFACTS_DIR:", A2_ARTIFACTS_DIR)
print("A2_EMB_DIR:", A2_EMB_DIR)

# -----------------------------
# 2) Utilidades: encontrar el último archivo por timestamp
# -----------------------------
# En Avance 2 se guardaron archivos con un timestamp:
# - lumina_chunks_tabular_YYYYMMDD_HHMMSS.parquet
# - lumina_embeddings_YYYYMMDD_HHMMSS.npz
# - embedding_manifest_YYYYMMDD_HHMMSS.json

TS_RE = re.compile(r"(\d{8}_\d{6})")

def pick_latest_file(folder: str, prefix: str, suffix: str) -> str:
    """
    Selecciona el archivo más reciente en 'folder' que cumpla con prefix+timestamp+suffix.
    Si no encuentra timestamp, cae a orden por fecha de modificación.
    """
    if not os.path.exists(folder):
        raise FileNotFoundError(f"No existe la carpeta: {folder}")

    candidates = []
    for fn in os.listdir(folder):
        if fn.startswith(prefix) and fn.endswith(suffix):
            m = TS_RE.search(fn)
            ts = m.group(1) if m else None
            full = os.path.join(folder, fn)
            mtime = os.path.getmtime(full)
            candidates.append((ts, mtime, full))

    if not candidates:
        raise FileNotFoundError(f"No encontré archivos {prefix}*{suffix} en: {folder}")

    # Si hay timestamps, ordenamos por timestamp (lexicográfico funciona con YYYYMMDD_HHMMSS)
    with_ts = [c for c in candidates if c[0] is not None]
    if with_ts:
        with_ts.sort(key=lambda x: x[0])  # asc
        return with_ts[-1][2]

    # Si no hay timestamps, ordenamos por mtime
    candidates.sort(key=lambda x: x[1])
    return candidates[-1][2]


tabular_path = pick_latest_file(A2_ARTIFACTS_DIR, prefix="lumina_chunks_tabular", suffix=".parquet")
embeddings_path = pick_latest_file(A2_EMB_DIR, prefix="lumina_embeddings", suffix=".npz")
manifest_path = pick_latest_file(A2_EMB_DIR, prefix="embedding_manifest_", suffix=".json")

print("\nÚltimo dataset tabular (.parquet):", os.path.basename(tabular_path))
print("Último archivo de embeddings (.npz):", os.path.basename(embeddings_path))
print("Último manifest (.json):", os.path.basename(manifest_path))

# -----------------------------
# 3) Carga de artefactos
# -----------------------------
df_chunks = pd.read_parquet(tabular_path)

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

# Carga de embeddings:
# Nota: a veces chunk_id queda como dtype object; np.load puede fallar si allow_pickle=False.
# Por seguridad, usamos allow_pickle=True solo para chunk_id (no debería incluir objetos peligrosos),
# y porque proviene de nuestros propios archivos.
emb_data = np.load(embeddings_path, allow_pickle=True)

# Las llaves esperadas (según Avance 2)
print("\nLlaves disponibles en el .npz:", emb_data.files)

chunk_ids_emb = emb_data["chunk_id"]
emb_rv = emb_data["emb_rv1909"]
emb_vbl = emb_data["emb_vbl"]

print("\nShape dataset tabular:", df_chunks.shape)
print("Número de chunks en tabular:", df_chunks["chunk_id"].nunique())

print("\nShape embeddings RV1909:", emb_rv.shape)
print("Shape embeddings VBL:", emb_vbl.shape)

print("\nChunks según manifest:", manifest.get("n_chunks"))

# -----------------------------
# 4) Sanity checks de consistencia y alineación
# -----------------------------

# (A) Verificar que el número de filas tabulares coincide con n_chunks del manifest
n_tab = int(df_chunks.shape[0])
n_manifest = int(manifest.get("n_chunks", -1))
if n_manifest != -1 and n_tab != n_manifest:
    print(f"Advertencia: tabular ({n_tab}) != manifest ({n_manifest})")
else:
    print("Conteo tabular vs manifest: OK")

# (B) Verificar shapes de embeddings contra tabular
if emb_rv.shape[0] != n_tab or emb_vbl.shape[0] != n_tab:
    print("Advertencia: embeddings no alinean en número de filas con tabular")
else:
    print("Número de embeddings coincide con #chunks tabular")

# (C) Verificar alineación por IDs (set equality)
ids_tabular = set(df_chunks["chunk_id"].astype(str))
ids_embeddings = set(chunk_ids_emb.astype(str))

aligned = (ids_tabular == ids_embeddings)
print("\n¿IDs alineados?:", aligned)
print("Diferencia (si existe):", len(ids_tabular.symmetric_difference(ids_embeddings)))

# (D) Verificar que estén en el mismo orden (esto importa para recuperar el embedding correcto)
same_order = np.array_equal(df_chunks["chunk_id"].astype(str).to_numpy(), chunk_ids_emb.astype(str))
print("¿Mismo orden (tabular vs embeddings)?:", same_order)

if not same_order:
    # Creamos un índice para reordenar embeddings según el orden del dataframe tabular.
    # Esto garantiza correspondencia 1 a 1: df_chunks.iloc[i] <-> emb[i]
    idx_map = {cid: i for i, cid in enumerate(chunk_ids_emb.astype(str))}
    reorder_idx = df_chunks["chunk_id"].astype(str).map(idx_map).to_numpy()

    emb_rv = emb_rv[reorder_idx]
    emb_vbl = emb_vbl[reorder_idx]
    chunk_ids_emb = chunk_ids_emb[reorder_idx]

    print("Embeddings reordenados para alinear con el orden del dataset tabular.")
    print("¿Mismo orden ahora?:", np.array_equal(df_chunks["chunk_id"].astype(str).to_numpy(),
                                               chunk_ids_emb.astype(str)))

# -----------------------------
# 5) Revisión rápida del dataset (columnas y ejemplos)
# -----------------------------
print("\nColumnas en df_chunks:")
print(df_chunks.columns.tolist())

display_cols = [
    "chunk_id", "chunk_ref", "book_es", "testament", "canon_group", "chapter",
    "verse_start", "verse_end"
]
text_cols = ["chunk_text_rv1909", "chunk_text_vbl"]

# Algunas columnas podrían no existir dependiendo de cómo guardaron el parquet.
existing_display_cols = [c for c in display_cols if c in df_chunks.columns]
existing_text_cols = [c for c in text_cols if c in df_chunks.columns]

print("\nEjemplo de 3 chunks (metadatos):")
display(df_chunks[existing_display_cols].sample(3, random_state=42))

print("\nEjemplo de 1 chunk (texto truncado):")
sample_row = df_chunks.sample(1, random_state=7)
for col in existing_text_cols:
    txt = sample_row[col].iloc[0]
    print(f"\n--- {col} ---")
    print((txt[:400] + " ...") if isinstance(txt, str) and len(txt) > 400 else txt)

# -----------------------------
# 6) (Opcional) Guardar una copia local del baseline dataset en Avance 3
# -----------------------------
# Esto  ayuda a reproducibilidad del Avance 3 sin depender del Avance 2.
# Aquí NO guardamos embeddings para no duplicar peso; solo tabular y un "pointer" al npz.
A4_ARTIFACTS_DIR = os.path.join(A4_DIR, "artifacts_baseline")
os.makedirs(A4_ARTIFACTS_DIR, exist_ok=True)

baseline_tabular_copy = os.path.join(A4_ARTIFACTS_DIR, f"baseline_chunks_tabular_copy_{datetime.now().strftime('%Y%m%d_%H%M%S')}.parquet")
df_chunks.to_parquet(baseline_tabular_copy, index=False)

print("\nCopia del dataset tabular guardada en:", baseline_tabular_copy)

A4_DIR: /content/drive/MyDrive/Proyecto Integrador/data
A2_DIR: /content/drive/MyDrive/Proyecto Integrador/
A2_ARTIFACTS_DIR: /content/drive/MyDrive/Proyecto Integrador/artifacts
A2_EMB_DIR: /content/drive/MyDrive/Proyecto Integrador/embeddings

Último dataset tabular (.parquet): lumina_chunks_tabular.parquet
Último archivo de embeddings (.npz): lumina_embeddings.npz
Último manifest (.json): embedding_manifest_20260207_084138.json

Llaves disponibles en el .npz: ['chunk_id', 'emb_rv1909', 'emb_vbl']

Shape dataset tabular: (10058, 29)
Número de chunks en tabular: 10058

Shape embeddings RV1909: (10058, 384)
Shape embeddings VBL: (10058, 384)

Chunks según manifest: 10058
Conteo tabular vs manifest: OK
Número de embeddings coincide con #chunks tabular

¿IDs alineados?: True
Diferencia (si existe): 0
¿Mismo orden (tabular vs embeddings)?: True

Columnas en df_chunks:
['chunk_id', 'book', 'book_es', 'chapter', 'verse_start', 'verse_end', 'chunk_ref', 'chunk_text_rv1909', 'chunk_text_vbl',

,chunk_id,chunk_ref,book_es,chapter,verse_start,verse_end
8292,chk_008292,PSA 9:13-14,Salmos,9,13,14
3127,chk_003127,EZK 29:9-12,Ezequiel,29,9,12
8507,chk_008507,PSA 37:3-4,Salmos,37,3,4



Ejemplo de 1 chunk (texto truncado):

--- chunk_text_rv1909 ---
Alma mía, en Dios solamente reposa; porque de él es mi esperanza. El solamente es mi fuerte y mí salud: es mi refugio, no resbalaré.

--- chunk_text_vbl ---
Solo en Dios encuentro paz. Mi esperanza viene de Él. Él es mi protector y salvador. Me guarda y por ello nunca estaré en peligro.

Copia del dataset tabular guardada en: /content/drive/MyDrive/Proyecto Integrador/data/artifacts_baseline/baseline_chunks_tabular_copy_20260223_033044.parquet


## **Interpretación de resultados (Parte 1)**

La verificación de artefactos del Avance 2 confirma que la base técnica para construir el Baseline es sólida y consistente:

- Se cargaron correctamente los tres activos fundamentales:
  - Dataset tabular: `10058` chunks.
  - Embeddings RV1909: shape `(10058, 384)`.
  - Embeddings VBL: shape `(10058, 384)`.
  - Manifest con trazabilidad consistente (`10058` chunks).

- El número de embeddings coincide exactamente con el número de filas del dataset tabular.

- La verificación por `chunk_id` confirmó:
  - Correspondencia 1 a 1 entre dataset y embeddings.
  - Sin diferencias en conjuntos de IDs.
  - Mismo orden entre dataframe y matrices vectoriales.

Esto es crítico porque garantiza que cada embedding representa exactamente el texto correspondiente del chunk, evitando errores silenciosos en retrieval.

**Conclusión:**  
Los artefactos del Avance 2 son consistentes, reproducibles y listos para ser indexados.  
Podemos avanzar con confianza a la construcción de los mecanismos de recuperación (retrieval) que definirán el Baseline.

---

> *"Llámame y te responderé; y te mostraré cosas grandes y ocultas que tú no conoces."*
>
> — Jeremías 33:3

---

# **Parte 2 — Construcción de los mecanismos de recuperación (Retrieval)**

En esta sección implementamos los tres enfoques de recuperación que conformarán nuestro Baseline:

1. **BM25 (Lexical Retrieval):**
   Basado en coincidencia de términos. Evalúa relevancia según frecuencia y rareza de palabras.

2. **Dense Retrieval (Semantic Retrieval):**
   Basado en similitud coseno entre embeddings vectoriales previamente generados.

3. **Híbrido (a implementar después):**
   Combinación ponderada de ambos enfoques.

El objetivo en esta etapa no es optimizar hiperparámetros, sino establecer un marco funcional y reproducible que permita comparar estrategias.


## **2.1 — Construcción de BM25**

Primero instalamos rank-bm25 si no está.


In [ ]:
# ============================================================
# Parte 2.1 — Construcción del índice BM25
# ============================================================

# Instalación (solo si es necesario)

!pip install rank-bm25 -q
from rank_bm25 import BM25Okapi

# -----------------------------
# 1) Tokenización simple
# -----------------------------
def simple_tokenizer(text):
    text = text.lower()
    text = re.sub(r"[^a-záéíóúñü\s]", " ", text)
    tokens = text.split()
    return tokens

# Usaremos la versión VBL para BM25 (más moderna en español)
corpus_texts = df_chunks["chunk_text_vbl"].astype(str).tolist()

tokenized_corpus = [simple_tokenizer(doc) for doc in corpus_texts]

# -----------------------------
# 2) Construcción del índice
# -----------------------------
bm25 = BM25Okapi(tokenized_corpus)

print("Índice BM25 construido correctamente.")
print("Número de documentos indexados:", len(tokenized_corpus))

Índice BM25 construido correctamente.
Número de documentos indexados: 10058


---

> *"Además de ser sabio, el Maestro impartió conocimientos a la gente. Escuchó, investigó y ordenó con cuidado muchos proverbios."*
>
> — Eclesiastés 12:9

---

### **Interpretación (BM25)**

El índice BM25 fue construido exitosamente sobre los `10058` chunks del corpus.

Este modelo léxico permitirá recuperar pasajes basándose en coincidencias explícitas de palabras clave.  
Servirá como punto de comparación frente al retrieval semántico (embeddings).

Limitación conocida:  
BM25 no captura sinonimia ni relaciones semánticas profundas.  
Por ejemplo, "angustia" y "ansiedad" podrían no coincidir si no comparten tokens.


## **2.2 — Preparación del Retrieval Denso**

Aquí no vamos a usar Chroma todavía. Primero lo haremos matemáticamente correcto y limpio usando similitud coseno directa.

Esto nos permite:

- Control total
- Evaluación clara
- Sin dependencias innecesarias
- Más alineado a CRISP-ML(Q)


In [ ]:
# ============================================================
# Parte 2.2 — Preparación del Retrieval Denso (Cosine Similarity)
# ============================================================

from sklearn.preprocessing import normalize

# Usaremos embeddings VBL para retrieval semántico
emb_matrix = emb_vbl.copy()

# Normalizamos para que el producto punto sea equivalente a coseno
emb_matrix = normalize(emb_matrix, norm="l2")

print("Shape matriz embeddings normalizada:", emb_matrix.shape)
print("Norma de primer vector (debería ser ~1):", np.linalg.norm(emb_matrix[0]))

Shape matriz embeddings normalizada: (10058, 384)
Norma de primer vector (debería ser ~1): 1.0


## **Interpretación (Dense Retrieval Setup)**

Se normalizó la matriz de embeddings para permitir el uso eficiente del producto punto como medida de similitud coseno.

Esto es importante porque:

- La similitud coseno mide orientación semántica.
- Permite recuperar versículos por cercanía conceptual.
- Evita que magnitudes influyan indebidamente en el ranking.

Con esta preparación, ya podemos implementar funciones de consulta para comparar:

- BM25
- Dense Retrieval

## **2.3 — Funciones de recuperación (BM25 vs Dense) y experimento inicial**

En esta subsección implementamos dos funciones de recuperación para comparar estrategias:

1. **BM25 (léxico):** recupera chunks por coincidencia de palabras clave.
2. **Dense Retrieval (semántico):** recupera chunks por similitud coseno en el espacio de embeddings.

Para asegurar comparabilidad:
- Ambas funciones devolverán un **DataFrame** con:
  - `rank`, `score`, `chunk_id`, `chunk_ref`, metadatos canónicos y texto.
- Usaremos **la traducción VBL** tanto para BM25 como para Dense Retrieval en este baseline (consistencia del corpus).

Finalmente, ejecutaremos un **experimento inicial** con consultas de prueba para observar diferencias cualitativas entre ambos enfoques.


In [ ]:
# ============================================================
# Parte 2.3 — Retrieval BM25 vs Dense + experimento inicial
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# 1) Helper: construir DataFrame de resultados
# -----------------------------
def build_results_df(indices, scores, method_name, query, top_k=5):
    """
    Construye un DataFrame con los top_k resultados y metadatos relevantes.
    """
    rows = []
    for r, (idx, sc) in enumerate(zip(indices[:top_k], scores[:top_k]), start=1):
        row = df_chunks.iloc[int(idx)]
        rows.append({
            "method": method_name,
            "query": query,
            "rank": r,
            "score": float(sc),
            "chunk_id": row["chunk_id"],
            "chunk_ref": row["chunk_ref"],
            "book_es": row["book_es"],
            "chapter": int(row["chapter"]),
            "verse_start": int(row["verse_start"]),
            "verse_end": int(row["verse_end"]),
            "text_vbl": row["chunk_text_vbl"],
        })
    return pd.DataFrame(rows)


# -----------------------------
# 2) Retrieval BM25
# -----------------------------
def retrieve_bm25(query, top_k=5):
    """
    Recuperación BM25: tokeniza la query y calcula scores BM25 sobre el corpus tokenizado.
    """
    q_tokens = simple_tokenizer(query)
    scores = bm25.get_scores(q_tokens)

    # Índices ordenados descendentemente por score
    top_idx = np.argsort(scores)[::-1][:top_k]
    top_scores = scores[top_idx]

    return build_results_df(top_idx, top_scores, "bm25", query, top_k=top_k)


# -----------------------------
# 3) Retrieval Dense (cosine similarity)
# -----------------------------
# Para generar embeddings de queries usaremos SentenceTransformers con el mismo modelo del Avance 2.
# Si ya está instalado, solo cargará el modelo; si no, primero lo instala.

!pip install sentence-transformers -q

from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
st_model = SentenceTransformer(MODEL_NAME)

def embed_query(query: str):
    """
    Genera embedding normalizado (L2) para una query en español.
    """
    vec = st_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    return vec  # shape (1, d)


def retrieve_dense(query, top_k=5):
    """
    Dense retrieval: cosine similarity entre embedding de query y matriz emb_matrix (normalizada).
    Como emb_matrix está normalizada, usar dot product equivale a coseno.
    """
    q_vec = embed_query(query)  # (1, d)
    sims = np.dot(emb_matrix, q_vec.T).reshape(-1)  # (n,)

    top_idx = np.argsort(sims)[::-1][:top_k]
    top_scores = sims[top_idx]

    return build_results_df(top_idx, top_scores, "dense", query, top_k=top_k)


# -----------------------------
# 4) Experimento inicial: queries de prueba
# -----------------------------
test_queries = [
    "Estoy muy ansioso y tengo miedo, ¿qué dice la Biblia sobre la paz?",
    "Me siento solo y sin esperanza, ¿hay algún pasaje de consuelo?",
    "Quiero sabiduría para tomar buenas decisiones",
    "Tengo muchas ocupaciones y responsabilidades, me siento frustrado , siento que no soy suficientemente capaz."
]

results = []
for q in test_queries:
    results.append(retrieve_bm25(q, top_k=5))
    results.append(retrieve_dense(q, top_k=5))

df_compare = pd.concat(results, ignore_index=True)

# ============================================================
# 5) Visualización amigable de resultados (sin truncar)
# ============================================================

pd.set_option("display.max_colwidth", 300)  # longitud del texto visible

def pretty_print_results(df_res, top_k=5, max_chars=450):
    """
    Imprime resultados agrupados por query y método.
    """
    for q in df_res["query"].unique():
        print("="*110)
        print("QUERY:", q)
        print("="*110)

        for method in ["bm25", "dense"]:
            sub = df_res[(df_res["query"] == q) & (df_res["method"] == method)].copy()
            sub = sub.sort_values("rank").head(top_k)

            print(f"\n--- {method.upper()} (Top-{top_k}) ---")
            for _, r in sub.iterrows():
                txt = r["text_vbl"]
                if isinstance(txt, str) and len(txt) > max_chars:
                    txt = txt[:max_chars] + " ..."

                print(f"\n#{int(r['rank'])} | score={r['score']:.4f} | {r['chunk_ref']} | {r['book_es']} {r['chapter']}:{r['verse_start']}-{r['verse_end']}")
                print(txt)

pretty_print_results(df_compare, top_k=5, max_chars=450)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUERY: Estoy muy ansioso y tengo miedo, ¿qué dice la Biblia sobre la paz?

--- BM25 (Top-5) ---

#1 | score=21.7935 | LUK 12:49-51 | Lucas 12:49-51
¡Yo he venido a prenderle fuego a la tierra, y en realidad desearía que ya estuviera ardiendo! ¡Pero tengo un bautismo por el cual pasar, y estoy en agonía, deseando que ya termine! ¿Ustedes creen que vine a traer paz a la tierra? No, les aseguro que traigo división.

#2 | score=21.2928 | JHN 14:25-27 | Juan 14:25-27
“Les estoy explicando esto ahora, mientras aún estoy con ustedes. Pero cuando el Padre envíe al Consolador, el Espíritu Santo, en mi lugar, él les enseñará todas las cosas y les recordará todo lo que yo les dije. “Yo les dejo paz; les estoy dando mi paz. La paz que yo les doy no se asemeja a ninguna cosa que ofrezca el mundo. No dejen que sus mentes estén ansiosas, y no tengan miedo.

#3 | score=20.8118 | REV 1:17-18 | Apocalipsis 1:17-18
Cuando lo vi, caí a sus pies como muerto. Pero él me tocó con su mano derecha y dijo: “No 

### **Interpretación (2.3 — Comparación BM25 vs Dense Retrieval)**

La comparación entre estrategias es concluyente:

- **Dense Retrieval (semántico)** mostró alta alineación con la intención del usuario en consultas emocionales.  
  Por ejemplo, ante “ansiedad/miedo/paz”, recuperó como top-1 *Filipenses 4:7-9*, un pasaje explícitamente asociado a paz y ausencia de ansiedad, lo cual sugiere que los embeddings capturan relaciones semánticas profundas.

- **BM25 (léxico)** fue competitivo cuando la consulta incluye términos explícitos presentes en el texto bíblico.  
  Sin embargo, también mostró un caso típico de debilidad: ante la misma consulta de “paz”, recuperó como top-1 *Lucas 12:49-51*, donde aparece la palabra “paz” pero el significado del pasaje es contrario (“no vine a traer paz”), evidenciando el riesgo de coincidencia literal sin comprensión semántica.

- En una consulta de “consuelo/esperanza”, los resultados sugieren señales complementarias:
  - **BM25** recupera pasajes que incluyen explícitamente “consuelo” (p. ej., *Salmos 119:81-82*), funcionando bien cuando el usuario usa vocabulario cercano al texto.
  - **Dense** recupera pasajes conceptualmente cercanos al estado emocional del usuario (dolor/desesperanza; p. ej. Job), lo cual puede ser semánticamente correcto, pero no siempre el “mejor primer contexto pastoral”.

- Para “sabiduría y buenas decisiones”, ambos enfoques convergen correctamente hacia Proverbios, mostrando consistencia del corpus y la ingeniería previa.

**Conclusión preliminar:**  
BM25 y Dense capturan señales distintas y complementarias: BM25 aporta anclaje literal y Dense aporta generalización semántica. Esto justifica implementar un **Baseline híbrido**, combinando ambas puntuaciones para reducir falsos positivos léxicos (como el caso de Lucas 12) y mantener precisión cuando el usuario usa términos del dominio.

## **2.4 — Retrieval híbrido (BM25 + Dense) como Baseline fuerte**

En esta subsección implementamos un **retrieval híbrido** para combinar las ventajas de:

- **BM25 (léxico):** excelente cuando el usuario utiliza vocabulario bíblico explícito (“consuelo”, “sabiduría”, “paz”, etc.).
- **Dense Retrieval (semántico):** robusto ante sinonimia y lenguaje cotidiano, capturando mejor la intención.

### **Idea central**
Calculamos dos puntuaciones por chunk:

- `bm25_score` (raw) → lo **normalizamos** a \[0,1\]
- `dense_score` (cosine) → ya está en \[-1,1\], lo **re-escalamos** a \[0,1\]

Luego combinamos:

\[
\text{hybrid} = \alpha \cdot \text{dense}_{norm} + (1-\alpha)\cdot \text{bm25}_{norm}
\]

donde `α` controla el peso semántico:
- `α` alto → más semántico
- `α` bajo → más literal

### **Evaluación rápida**
Probamos varios valores de `α` (0.3, 0.5, 0.7) sobre las mismas queries de prueba y comparamos los **Top-K** pasajes recuperados.

> Nota: Este híbrido sigue siendo un **baseline**, no es optimización final; su objetivo es proveer un marco de referencia sólido para iteraciones posteriores.

In [ ]:
# ============================================================
# 2.4 — Retrieval híbrido (BM25 + Dense)
# ============================================================

# -----------------------------
# 1) Normalización de scores
# -----------------------------
def minmax_norm(arr: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    Normaliza un vector a [0,1] con Min-Max.
    Maneja casos degenerados (max == min).
    """
    arr = np.asarray(arr, dtype=np.float32)
    a_min, a_max = float(arr.min()), float(arr.max())
    if abs(a_max - a_min) < eps:
        return np.zeros_like(arr)
    return (arr - a_min) / (a_max - a_min)


def dense_to_01(dense_scores: np.ndarray) -> np.ndarray:
    """
    Reescala coseno [-1, 1] a [0, 1] de forma determinista.
    """
    dense_scores = np.asarray(dense_scores, dtype=np.float32)
    return (dense_scores + 1.0) / 2.0


# -----------------------------
# 2) Retrieval híbrido
# -----------------------------
def retrieve_hybrid(query: str, top_k: int = 5, alpha: float = 0.5):
    """
    Recuperación híbrida:
    1) Calcula scores BM25 raw -> min-max a [0,1]
    2) Calcula scores dense (coseno) -> a [0,1] con (x+1)/2
    3) Combina: alpha*dense_norm + (1-alpha)*bm25_norm
    """
    # BM25
    q_tokens = simple_tokenizer(query)
    bm25_scores = bm25.get_scores(q_tokens)  # (n,)

    bm25_norm = minmax_norm(bm25_scores)

    # Dense
    q_vec = embed_query(query)  # (1,d)
    dense_scores = np.dot(emb_matrix, q_vec.T).reshape(-1)  # (n,)
    dense_norm = dense_to_01(dense_scores)

    # Combinación híbrida
    hybrid_scores = alpha * dense_norm + (1 - alpha) * bm25_norm

    # Top indices
    top_idx = np.argsort(hybrid_scores)[::-1][:top_k]
    top_scores = hybrid_scores[top_idx]

    # Construimos DF de resultados
    df_out = build_results_df(top_idx, top_scores, f"hybrid_a{alpha}", query, top_k=top_k)

    # Guardamos también scores individuales para trazabilidad/diagnóstico
    df_out["bm25_norm"] = bm25_norm[top_idx]
    df_out["dense_norm"] = dense_norm[top_idx]

    return df_out


# -----------------------------
# 3) Visualización bonita para híbrido
# -----------------------------
def pretty_print_results_hybrid(df_res, top_k=5, max_chars=450):
    """
    Imprime resultados híbridos, mostrando scores híbridos + contribuciones.
    """
    for q in df_res["query"].unique():
        print("="*110)
        print("QUERY:", q)
        print("="*110)

        for method in df_res[df_res["query"] == q]["method"].unique():
            sub = df_res[(df_res["query"] == q) & (df_res["method"] == method)].copy()
            sub = sub.sort_values("rank").head(top_k)

            print(f"\n--- {method.upper()} (Top-{top_k}) ---")
            for _, r in sub.iterrows():
                txt = r["text_vbl"]
                if isinstance(txt, str) and len(txt) > max_chars:
                    txt = txt[:max_chars] + " ..."

                bm = float(r.get("bm25_norm", np.nan))
                de = float(r.get("dense_norm", np.nan))

                print(
                    f"\n#{int(r['rank'])} | hybrid={r['score']:.4f} | bm25_norm={bm:.4f} | dense_norm={de:.4f} "
                    f"| {r['chunk_ref']} | {r['book_es']} {r['chapter']}:{r['verse_start']}-{r['verse_end']}"
                )
                print(txt)


# -----------------------------
# 4) Experimento: probar varios alpha
# -----------------------------
alphas = [0.3, 0.5, 0.7]
hybrid_results = []

for a in alphas:
    for q in test_queries:
        hybrid_results.append(retrieve_hybrid(q, top_k=5, alpha=a))

df_hybrid = pd.concat(hybrid_results, ignore_index=True)

print("Resultados Hybrid (Top-5) para distintos α:")
display(df_hybrid[["method","query","rank","score","bm25_norm","dense_norm","chunk_ref","book_es","chapter","verse_start","verse_end"]])

# Impresión tipo ficha (más fácil de leer)
pretty_print_results_hybrid(df_hybrid, top_k=5, max_chars=450)

Resultados Hybrid (Top-5) para distintos α:


,method,query,rank,score,bm25_norm,dense_norm,chunk_ref,book_es,chapter,verse_start,verse_end
0,hybrid_a0.3,"Estoy muy ansioso y tengo miedo, ¿qué dice la Biblia sobre la paz?",1,0.927952,1.000000,0.759840,LUK 12:49-51,Lucas,12,49,51
1,hybrid_a0.3,"Estoy muy ansioso y tengo miedo, ¿qué dice la Biblia sobre la paz?",2,0.913921,0.977026,0.766679,JHN 14:25-27,Juan,14,25,27
2,hybrid_a0.3,"Estoy muy ansioso y tengo miedo, ¿qué dice la Biblia sobre la paz?",3,0.865794,0.954952,0.657758,REV 1:17-18,Apocalipsis,1,17,18
3,hybrid_a0.3,"Estoy muy ansioso y tengo miedo, ¿qué dice la Biblia sobre la paz?",4,0.864181,0.907824,0.762348,ISA 52:5-8,Isaías,52,5,8
4,hybrid_a0.3,"Estoy muy ansioso y tengo miedo, ¿qué dice la Biblia sobre la paz?",5,0.856099,0.911484,0.726866,HEB 12:19-21,Hebreos,12,19,21
5,hybrid_a0.3,"Me siento solo y sin esperanza, ¿hay algún pasaje de consuelo?",1,0.885502,1.000000,0.618339,EPH 4:4-6,Efesios,4,4,6
6,hybrid_a0.3,"Me siento solo y sin esperanza, ¿hay algún pasaje de consuelo?",2,0.865093,0.911830,0.756039,PSA 119:81-82,Salmos,119,81,82
7,hybrid_a0.3,"Me siento solo y sin esperanza, ¿hay algún pasaje de consuelo?",3,0.831856,0.917542,0.631923,2CO 11:1-3,2 Corintios,11,1,3
8,hybrid_a0.3,"Me siento solo y sin esperanza, ¿hay algún pasaje de consuelo?",4,0.802341,0.839469,0.715711,PSA 62:5-6,Salmos,62,5,6
9,hybrid_a0.3,"Me siento solo y sin esperanza, ¿hay algún pasaje de consuelo?",5,0.757367,0.797996,0.662566,PSA 119:49-50,Salmos,119,49,50


QUERY: Estoy muy ansioso y tengo miedo, ¿qué dice la Biblia sobre la paz?

--- HYBRID_A0.3 (Top-5) ---

#1 | hybrid=0.9280 | bm25_norm=1.0000 | dense_norm=0.7598 | LUK 12:49-51 | Lucas 12:49-51
¡Yo he venido a prenderle fuego a la tierra, y en realidad desearía que ya estuviera ardiendo! ¡Pero tengo un bautismo por el cual pasar, y estoy en agonía, deseando que ya termine! ¿Ustedes creen que vine a traer paz a la tierra? No, les aseguro que traigo división.

#2 | hybrid=0.9139 | bm25_norm=0.9770 | dense_norm=0.7667 | JHN 14:25-27 | Juan 14:25-27
“Les estoy explicando esto ahora, mientras aún estoy con ustedes. Pero cuando el Padre envíe al Consolador, el Espíritu Santo, en mi lugar, él les enseñará todas las cosas y les recordará todo lo que yo les dije. “Yo les dejo paz; les estoy dando mi paz. La paz que yo les doy no se asemeja a ninguna cosa que ofrezca el mundo. No dejen que sus mentes estén ansiosas, y no tengan miedo.

#3 | hybrid=0.8658 | bm25_norm=0.9550 | dense_norm=0.6578 | 

## **Interpretación (Parte 2.4 — Retrieval híbrido)**

El retrieval híbrido (BM25 + Dense) busca combinar:

- **BM25 (léxico):** fuerte cuando hay coincidencias explícitas de palabras clave (“paz”, “consuelo”, “sabiduría”).
- **Dense (semántico):** fuerte cuando el usuario expresa intención en lenguaje cotidiano o con sinonimia.

Al evaluar distintos valores de **α** (0.3, 0.5, 0.7), observamos:

1. **Los resultados son consistentes y estables**: los pasajes recuperados no cambian drásticamente al variar α, lo cual sugiere que ambos métodos tienden a coincidir en “zonas” semánticas similares para este corpus.

2. **Caso relevante: “paz” (query sobre ansiedad/miedo)**  
   El pasaje **LUK 12:49-51** permanece como Top-1 incluso con α alto. Esto ocurre porque:
   - Tiene coincidencia léxica máxima (**bm25_norm = 1.0**) al contener explícitamente la palabra “paz”.
   - También recibe un puntaje semántico alto (**dense_norm ≈ 0.76**), pues el embedding detecta relación temática con “paz”, aunque el pasaje la mencione en un sentido de contraste (“no vine a traer paz”).
   
   Esto evidencia una limitación esperada del baseline: **los métodos de similitud (léxica y densa) pueden priorizar pasajes con alta coincidencia temática aunque el contenido no sea pastoralmente el más adecuado**, especialmente cuando existe negación o contraste semántico.

3. **Consultas de sabiduría**  
   En “tomar buenas decisiones”, tanto BM25 como Dense recuperan consistentemente pasajes de Proverbios (p. ej., PRO 5, PRO 8, PRO 2), mostrando que el baseline funciona bien cuando hay alta alineación temática y vocabulario consistente.

4. **Conclusión del baseline híbrido**  
   El híbrido ofrece un marco de referencia sólido, pero también deja claro que **la relevancia “pastoral” no siempre coincide con relevancia léxica o semántica**.  
   Esto justifica que, en iteraciones posteriores, se incorporen mecanismos de mejora como:
   - reglas simples de desambiguación (p. ej., penalizar negaciones explícitas en contextos de consuelo/paz), o
   - re-ranking más avanzado.

Como baseline para *retrieval*, proponemos mantener **α = 0.5** por equilibrio, documentando explícitamente la limitación observada (lexical trap por negación) como oportunidad de mejora en el siguiente avance.

## **2.4.1 (Sección de experimentación) Ajuste heurístico baseline (penalización por negación / contraste semántico)**

En la evaluación del retrieval híbrido observamos un caso clásico de *lexical trap*: algunos pasajes pueden contener la palabra clave objetivo (p. ej. “paz”) pero dentro de una estructura de **negación o contraste** (“no… paz”, “no vine a traer paz”, etc.).  
Tanto BM25 como Dense pueden asignar puntajes altos a estos chunks por coincidencia lexical y proximidad temática, aunque el pasaje no sea el más adecuado para responder una consulta pastoral de consuelo.

Para mantener este entregable como **baseline** (sin re-rankers complejos), implementamos un ajuste heurístico simple y explicable:

- Detectar patrones de negación/contraste en una ventana cercana a la palabra objetivo.
- Aplicar una penalización multiplicativa al score híbrido del chunk.
- Recalcular el Top-k con el score penalizado.

Esta heurística no pretende “resolver” semántica profunda, sino añadir una regla mínima que refleje un criterio de negocio:  
**si el usuario pide paz/consuelo, evitamos priorizar pasajes que discuten explícitamente “no paz” como mensaje central.**

In [ ]:
# ------------------------------------------------------------
# 1) Heurística: penalizar chunks con negación/contraste
#    cerca del término objetivo (ej. paz, consuelo, esperanza).
# ------------------------------------------------------------
NEGATION_WORDS = [
    r"\bno\b", r"\bnunca\b", r"\bjam[aá]s\b", r"\bni\b", r"\bsin\b"
]

def normalize_text(s: str) -> str:
    """Normaliza texto: minúsculas y espacios."""
    if s is None:
        return ""
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def has_negation_near_term(text: str, term: str, window_chars: int = 60) -> bool:
    """
    Devuelve True si detecta negación/contraste en una ventana cercana al término.
    Usamos una ventana por caracteres para mantenerlo simple y robusto.
    """
    t = normalize_text(text)
    term_n = normalize_text(term)

    idx = t.find(term_n)
    if idx == -1:
        return False

    left = max(0, idx - window_chars)
    right = min(len(t), idx + len(term_n) + window_chars)
    snippet = t[left:right]

    # Si aparece una palabra de negación dentro de la ventana, se activa penalización
    neg_pattern = "(" + "|".join(NEGATION_WORDS) + ")"
    return re.search(neg_pattern, snippet) is not None

def apply_negation_penalty(df_ranked: pd.DataFrame,
                           term: str,
                           penalty: float = 0.85,
                           text_col: str = "text_vbl") -> pd.DataFrame:
    """
    Agrega columnas:
      - negation_flag: bool
      - score_penalized: score ajustado
    y devuelve df re-rankeado por score_penalized.
    """
    df = df_ranked.copy()

    flags = []
    for txt in df[text_col].fillna("").tolist():
        flags.append(has_negation_near_term(txt, term))

    df["negation_flag"] = flags
    df["score_penalized"] = df["score"] * np.where(df["negation_flag"], penalty, 1.0)

    # Re-rank dentro de cada query
    df = df.sort_values(["query", "score_penalized"], ascending=[True, False]).copy()

    # Recalcular rank (1..k) por query
    df["rank_penalized"] = df.groupby("query").cumcount() + 1
    return df

# ------------------------------------------------------------
# 2) Definir términos objetivo por query
#    (baseline simple: reglas manuales por intención)
# ------------------------------------------------------------
# Asignamos un "término objetivo" por query para detectar negaciones cerca.
# Esto es deliberadamente simple y explicable.
QUERY_TERMS = {
    "Estoy muy ansioso y tengo miedo, ¿qué dice la Biblia sobre la paz?": "paz",
    "Me siento solo y sin esperanza, ¿hay algún pasaje de consuelo?": "consuelo",
    "Quiero sabiduría para tomar buenas decisiones": "sabiduría",
    "Tengo muchas ocupaciones y responsabilidades, me siento frustrado , siento que no soy suficientemente capaz.": "paz"
}

# ------------------------------------------------------------
# 3) Ejecutar penalización sobre el resultado híbrido
df_hybrid_all = df_hybrid
assert "df_hybrid_all" in globals(), "No encuentro df_hybrid_all. Asegúrate de haber corrido la sección 2.4 y guardado resultados."

# Parámetros de penalización
NEG_PENALTY = 0.80        # penalización un poco más fuerte para que tenga efecto visible
WINDOW_CHARS = 70         # ventana algo más amplia
TEXT_COL = "text_vbl"     # usamos VBL por consistencia con retrieval que estás comparando

dfs = []
for q, term in QUERY_TERMS.items():
    sub = df_hybrid_all[df_hybrid_all["query"] == q].copy()
    if sub.empty:
        continue

    # Ajuste: inyectar la ventana como parámetro en la función de detección
    # (para no reescribirla, hacemos una versión inline)
    sub2 = sub.copy()
    flags = []
    for txt in sub2[TEXT_COL].fillna("").tolist():
        flags.append(has_negation_near_term(txt, term, window_chars=WINDOW_CHARS))

    sub2["negation_flag"] = flags
    sub2["score_penalized"] = sub2["score"] * np.where(sub2["negation_flag"], NEG_PENALTY, 1.0)

    # Re-rank por query y por método (porque tienes hybrid_a0.3, hybrid_a0.5, hybrid_a0.7)
    sub2 = sub2.sort_values(["method", "query", "score_penalized"], ascending=[True, True, False]).copy()
    sub2["rank_penalized"] = sub2.groupby(["method", "query"]).cumcount() + 1

    dfs.append(sub2)

df_hybrid_penalized = pd.concat(dfs, ignore_index=True)

# ------------------------------------------------------------
# 4) Mostrar comparativo Top-5 antes vs después (por query)
# ------------------------------------------------------------
def show_before_after(query: str, method: str, topk: int = 5):
    before = df_hybrid_all[(df_hybrid_all["query"] == query) & (df_hybrid_all["method"] == method)] \
        .sort_values("rank").head(topk)[["rank","score","chunk_ref","book_es","chapter","verse_start","verse_end",TEXT_COL]]

    after = df_hybrid_penalized[(df_hybrid_penalized["query"] == query) & (df_hybrid_penalized["method"] == method)] \
        .sort_values("rank_penalized").head(topk)[["rank_penalized","score","score_penalized","negation_flag","chunk_ref","book_es","chapter","verse_start","verse_end",TEXT_COL]]

    print("="*110)
    print(f"QUERY: {query}")
    print(f"METHOD: {method}")
    print("="*110)

    print("\n--- ANTES (Top-k por score híbrido) ---\n")
    for _, r in before.iterrows():
        ref = f"{r['chunk_ref']} | {r['book_es']} {int(r['chapter'])}:{int(r['verse_start'])}-{int(r['verse_end'])}"
        print(f"#{int(r['rank'])} | score={r['score']:.4f} | {ref}")
        print(str(r[TEXT_COL])[:350], "...\n")

    print("\n--- DESPUÉS (Top-k por score penalizado) ---\n")
    for _, r in after.iterrows():
        ref = f"{r['chunk_ref']} | {r['book_es']} {int(r['chapter'])}:{int(r['verse_start'])}-{int(r['verse_end'])}"
        flag = "NEG" if r["negation_flag"] else "OK "
        print(f"#{int(r['rank_penalized'])} | score_pen={r['score_penalized']:.4f} | orig={r['score']:.4f} | {flag} | {ref}")
        print(str(r[TEXT_COL])[:350], "...\n")

# Ejemplo focal:
show_before_after(
    "Estoy muy ansioso y tengo miedo, ¿qué dice la Biblia sobre la paz?",
    method="hybrid_a0.5",
    topk=5
)

QUERY: Estoy muy ansioso y tengo miedo, ¿qué dice la Biblia sobre la paz?
METHOD: hybrid_a0.5

--- ANTES (Top-k por score híbrido) ---

#1 | score=0.8799 | LUK 12:49-51 | Lucas 12:49-51
¡Yo he venido a prenderle fuego a la tierra, y en realidad desearía que ya estuviera ardiendo! ¡Pero tengo un bautismo por el cual pasar, y estoy en agonía, deseando que ya termine! ¿Ustedes creen que vine a traer paz a la tierra? No, les aseguro que traigo división. ...

#2 | score=0.8719 | JHN 14:25-27 | Juan 14:25-27
“Les estoy explicando esto ahora, mientras aún estoy con ustedes. Pero cuando el Padre envíe al Consolador, el Espíritu Santo, en mi lugar, él les enseñará todas las cosas y les recordará todo lo que yo les dije. “Yo les dejo paz; les estoy dando mi paz. La paz que yo les doy no se asemeja a ninguna cosa que ofrezca el mundo. No dejen que sus mente ...

#3 | score=0.8351 | ISA 52:5-8 | Isaías 52:5-8
¿Qué tengo que hacer ahora? pregunta el Señor. Mi pueblo ha sido llevado al cautiverio si

### **Interpretación (2.4.1 — Penalización por negación / contraste)**

Al aplicar la penalización heurística por “negación cercana al término objetivo”, observamos que el mecanismo **sí funciona técnicamente**: pasajes como *Lucas 12:49–51* (que incluye explícitamente “no vine a traer paz”) fueron marcados con `negation_flag=True` y descendieron en el ranking al reducir su `score_penalized`.

Sin embargo, este experimento también revela una limitación crítica del enfoque:

- La regla es **demasiado superficial**: detecta la palabra “no” cerca del término objetivo, pero **no distingue** entre una negación que invalida el sentido del pasaje (“no… paz” como contraste) y una negación pastoralmente útil (“No dejen que sus mentes estén ansiosas…”).
- En consecuencia, el método también penalizó pasajes altamente pertinentes como *Juan 14:25–27*, donde la negación aparece como exhortación al creyente y no como contradicción del concepto de paz.

Por lo tanto, esta heurística se conserva únicamente como **demostración baseline** de un fenómeno real en retrieval: los *lexical traps* y la sensibilidad del ranking a reglas simples.  
En iteraciones futuras (Avance 4+), la solución adecuada sería incorporar mecanismos de **re-ranking semántico** o criterios de evaluación más robustos (p. ej. judge LLM con trazabilidad, o re-rankers entrenados), en lugar de reglas basadas en patrones de negación.

---

> *"Pero todo lo que se pone bajo la luz se hace visible, y lo que se hace visible se convierte en luz."*
>
> — Efesios 5:13-14

---

# **Parte 3 – Evaluación formal de los modelos**

En esta sección realizamos una evaluación cuantitativa controlada de los métodos de recuperación implementados:

- BM25 (lexical)
- Dense Retrieval (semántico)
- Hybrid (combinación de ambos)

El objetivo es medir desempeño usando métricas estándar de Information Retrieval:

- Recall@k
- MRR (Mean Reciprocal Rank)

Para garantizar reproducibilidad:

- Se reutilizan los embeddings generados en Avance 2.
- Se utiliza el mismo modelo SentenceTransformer empleado en la Parte 2.
- Se implementan funciones estandarizadas de retrieval con interfaz uniforme.

Esta evaluación permitirá justificar técnicamente cuál estrategia es más adecuada como baseline del sistema LUMINA.

## **3.1 Funciones Limpias de Retrieval**

**Nota**: el modelo SentenceTransformer se inicializó previamente como st_model en la sección 2.3.



In [ ]:
# ==========================================
# 3.1 Funciones estandarizadas de retrieval
# ==========================================

from sklearn.preprocessing import MinMaxScaler

# ---------------------------------------------------
# 1. Cosine similarity
# ---------------------------------------------------

def cosine_similarity_matrix(M, v):
    """
    M: (N, d)
    v: (d,)
    returns: (N,)
    """
    v_norm = np.linalg.norm(v)
    M_norm = np.linalg.norm(M, axis=1)
    return (M @ v) / (M_norm * v_norm + 1e-12)


# ---------------------------------------------------
# 2. BM25 Retrieval
# ---------------------------------------------------

def bm25_retrieve(query, top_k=5):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)

    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_idx:
        results.append({
            "chunk_ref": df_chunks.iloc[idx]["chunk_ref"],
            "score": float(scores[idx])
        })
    return results


# ---------------------------------------------------
# 3. Dense Retrieval (usando emb_vbl)
# ---------------------------------------------------

def dense_retrieve(query, top_k=5, embeddings=emb_vbl):

    query_emb = st_model.encode([query])[0]
    sims = cosine_similarity_matrix(embeddings, query_emb)

    top_idx = np.argsort(sims)[::-1][:top_k]

    results = []
    for idx in top_idx:
        results.append({
            "chunk_ref": df_chunks.iloc[idx]["chunk_ref"],
            "score": float(sims[idx])
        })
    return results


# ---------------------------------------------------
# 4. Hybrid Retrieval
# ---------------------------------------------------

def hybrid_retrieve(query, alpha=0.5, top_k=5, embeddings=emb_vbl):

    # BM25 scores
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query).reshape(-1, 1)

    # Dense scores
    query_emb = st_model.encode([query])[0]
    dense_scores = cosine_similarity_matrix(embeddings, query_emb).reshape(-1, 1)

    # Normalización
    scaler = MinMaxScaler()
    bm25_norm = scaler.fit_transform(bm25_scores).flatten()

    scaler = MinMaxScaler()
    dense_norm = scaler.fit_transform(dense_scores).flatten()

    hybrid_scores = alpha * bm25_norm + (1 - alpha) * dense_norm

    top_idx = np.argsort(hybrid_scores)[::-1][:top_k]

    results = []
    for idx in top_idx:
        results.append({
            "chunk_ref": df_chunks.iloc[idx]["chunk_ref"],
            "score": float(hybrid_scores[idx])
        })

    return results


print("Funciones de retrieval listas.")

Funciones de retrieval listas.


---

> *"La balanza justa y los pesos exactos son del Señor; él es quien establece todas las pesas de la bolsa."*
>
> — Proverbios 16:11

---

### **Interpretación del código anterior**

Se implementaron tres funciones de recuperación con interfaz homogénea.
Cada método devuelve una lista de `chunk_ref` ordenados por relevancia.

Esto permite comparar estrategias bajo las mismas condiciones y preparar
la evaluación cuantitativa en la siguiente subsección.

## **3.2 Definición del Conjunto de Evaluación**

Aquí vamos a definir:

- Queries

- Ground truth manual (relevancia binaria)

In [ ]:
# ==========================================
# 3.2 Dataset de evaluación manual
# ==========================================

evaluation_queries = [
    "Estoy muy ansioso y tengo miedo, ¿qué dice la Biblia sobre la paz?",
    "Me siento solo y sin esperanza, ¿hay algún pasaje de consuelo?",
    "Quiero sabiduría para tomar buenas decisiones"
]

# Ground truth simplificado (pasajes seleccionados por sugerencias online)

ground_truth = {
    evaluation_queries[0]: ["JHN 14:25-27", "PHP 4:7-9", "PSA 62:5-6"],
    evaluation_queries[1]: ["PSA 119:81-82", "PSA 119:49-50"],
    evaluation_queries[2]: ["PRO 2:11-12", "PRO 5:1-2", "PRO 8:11-12"]
}

print("Queries y ground truth definidos.")

Queries y ground truth definidos.


## **3.3 Métricas: Recall@k y MRR**

In [ ]:
# ==========================================
# 3.3 Métricas de evaluación
# ==========================================

def recall_at_k(results, relevant_refs, k=5):
    retrieved_refs = [r["chunk_ref"] for r in results[:k]]
    hits = sum([1 for ref in retrieved_refs if ref in relevant_refs])
    return hits / len(relevant_refs)


def mrr(results, relevant_refs):
    for i, r in enumerate(results):
        if r["chunk_ref"] in relevant_refs:
            return 1 / (i + 1)
    return 0


def evaluate_method(method_name, retrieval_function, k=5):

    recall_scores = []
    mrr_scores = []

    for query in evaluation_queries:

        results = retrieval_function(query, top_k=k)
        relevant = ground_truth[query]

        recall_scores.append(recall_at_k(results, relevant, k))
        mrr_scores.append(mrr(results, relevant))

    return {
        "method": method_name,
        "Recall@5": np.mean(recall_scores),
        "MRR": np.mean(mrr_scores)
    }

## **3.4 Ejecutar Evaluación**

In [ ]:
results = []

results.append(evaluate_method("BM25", bm25_retrieve))
results.append(evaluate_method("Dense", dense_retrieve))
results.append(evaluate_method("Hybrid α=0.5", lambda q, top_k=5: hybrid_retrieve(q, alpha=0.5, top_k=top_k)))

import pandas as pd

df_metrics = pd.DataFrame(results)
df_metrics

,method,Recall@5,MRR
0,BM25,0.333333,0.333333
1,Dense,0.555556,0.666667
2,Hybrid α=0.5,0.500000,0.416667


### **Interpretación formal de df_metrics**

1. **BM25 (Retrieval Léxico)**
  
    El enfoque basado en coincidencia literal de términos muestra un desempeño limitado. Si bien puede capturar consultas donde existe coincidencia directa de palabras clave (ej. “sabiduría”), falla cuando la intención del usuario es implícita o emocional (ej. ansiedad, frustración, desesperanza).

    Esto confirma que la similitud léxica no es suficiente en dominios donde la carga semántica es más relevante que la coincidencia textual.

2. **Dense Retrieval (Embeddings Semánticos)**

    El modelo basado en embeddings mostró el mejor desempeño en ambas métricas:
    - Mayor Recall@5 → Recupera con mayor frecuencia el pasaje correcto dentro del Top-5.
    - Mayor MRR → El pasaje relevante tiende a aparecer en posiciones más altas.

    Esto indica que el espacio vectorial generado en el Avance 2 captura adecuadamente la semántica teológica y emocional del texto bíblico.

3. **Hybrid (α=0.5)**

    El enfoque híbrido mejora ligeramente el desempeño frente a BM25 puro, pero no supera al modelo denso.
    
    Esto sugiere que, en este dominio específico (consultas pastorales/emocionales), la señal semántica tiene mayor peso informativo que la coincidencia literal.

## **Conclusión de la Parte 3**

El modelo Dense Retrieval será adoptado como baseline oficial para la siguiente fase del proyecto (integración generativa), ya que demuestra mayor capacidad para recuperar pasajes bíblicos pertinentes ante consultas de naturaleza emocional y existencial.

Este resultado es coherente con la hipótesis central del proyecto:
en una arquitectura RAG aplicada al acompañamiento pastoral, la comprensión semántica es crítica para garantizar respuestas relevantes y responsables.

---

> *"Secóse la hierba, cayóse la flor; mas la palabra del Dios nuestro permanece para siempre."*
>
> — Isaías 40:8

---

## **3.5 Análisis Experimental de Sensibilidad del Baseline**

En esta subsección profundizamos en la evaluación del baseline mediante un análisis experimental de sensibilidad.

El objetivo es examinar:

- El impacto de diferentes valores de α en el método híbrido.

- La variación en desempeño entre Recall@3 y Recall@5.

- La estabilidad del modelo ante consultas más complejas y emocionalmente críticas.

Este análisis permite validar si el desempeño observado en la sección anterior es consistente o si depende fuertemente de la configuración de parámetros o del tipo de consulta.

In [ ]:
# =========================
# Paso 1 - Expansión del Conjunto de Evaluación
# =========================

evaluation_queries_extended = {
    "Estoy muy ansioso y tengo miedo, ¿qué dice la Biblia sobre la paz?": [
        "PHP 4:7-9", "PSA 62:5-6", "JHN 14:25-27"
    ],
    "Me siento solo y sin esperanza, ¿hay algún pasaje de consuelo?": [
        "PSA 119:49-50", "PSA 62:5-6", "JOB 17:15-16"
    ],
    "Quiero sabiduría para tomar buenas decisiones": [
        "PRO 2:11-12", "PRO 5:1-2", "PRO 8:11-12"
    ],
    "Siento que fracasé en la vida. He considerado suicidarme. ¿Qué me diría Dios ahora?": [
        "PSA 34:18", "ROM 8:38-39", "ISA 41:10"
    ],
    "Vivo en constante frustración y depresión. Mi corazón está entristecido. ¿Qué puedo hacer?": [
        "PSA 42:11", "MAT 11:28-30", "2CO 1:3-4"
    ],
    "¿Qué dice la Biblia sobre la tecnología y la ciencia?": [
        "GEN 1:28", "PRO 25:2", "ECC 7:29"
    ],
    "Tengo culpa por errores del pasado. ¿Hay perdón para mí?": [
        "1JN 1:9", "PSA 103:12", "ISA 1:18"
    ],
    "Siento ira constante y rencor. ¿Cómo puedo cambiar?": [
        "EPH 4:31-32", "COL 3:13", "JAS 1:20"
    ]
}

In [ ]:
# =========================
# Paso 2 - Función flexible para evaluar Recall@K
# =========================

def evaluate_method_extended(method_name, retrieve_fn, queries_dict, k=5):

    recall_hits = 0
    reciprocal_ranks = []
    total_queries = len(queries_dict)

    for query, relevant_refs in queries_dict.items():

        results = retrieve_fn(query, top_k=k)
        retrieved_refs = [r["chunk_ref"] for r in results]

        # Recall@k
        if any(ref in retrieved_refs for ref in relevant_refs):
            recall_hits += 1

        # MRR
        rr = 0
        for rank, ref in enumerate(retrieved_refs, start=1):
            if ref in relevant_refs:
                rr = 1 / rank
                break
        reciprocal_ranks.append(rr)

    recall_at_k = recall_hits / total_queries
    mrr = sum(reciprocal_ranks) / total_queries

    return {
        "method": method_name,
        "k": k,
        "Recall@k": recall_at_k,
        "MRR": mrr
    }

In [ ]:
# =========================
# Paso 3 - Probar múltiples α y múltiples k
# =========================

alphas = [0.2, 0.5, 0.8]
ks = [3, 5]

results_experiments = []

# BM25 y Dense
for k in ks:
    results_experiments.append(
        evaluate_method_extended("BM25", bm25_retrieve, evaluation_queries_extended, k)
    )
    results_experiments.append(
        evaluate_method_extended("Dense", dense_retrieve, evaluation_queries_extended, k)
    )

# Hybrid con diferentes alphas
for alpha in alphas:
    for k in ks:
        results_experiments.append(
            evaluate_method_extended(
                f"Hybrid α={alpha}",
                lambda q, top_k=k: hybrid_retrieve(q, alpha=alpha, top_k=top_k),
                evaluation_queries_extended,
                k
            )
        )

import pandas as pd
df_experiments = pd.DataFrame(results_experiments)
df_experiments.sort_values(by=["k", "Recall@k"], ascending=False)

,method,k,Recall@k,MRR
3,Dense,5,0.375,0.3125
5,Hybrid α=0.2,5,0.250,0.2500
2,BM25,5,0.125,0.1250
7,Hybrid α=0.5,5,0.125,0.1250
9,Hybrid α=0.8,5,0.125,0.1250
1,Dense,3,0.375,0.3125
4,Hybrid α=0.2,3,0.250,0.2500
0,BM25,3,0.125,0.1250
6,Hybrid α=0.5,3,0.125,0.1250
8,Hybrid α=0.8,3,0.125,0.1250


## **Interpretación Global del Experimento**

Al ampliar el conjunto de evaluación con consultas:

- Más emocionales
- Más críticas (suicidio, culpa, depresión)
- Más abstractas (tecnología, ciencia)
- El desempeño general bajó respecto a la evaluación inicial.

Esto es normal y científicamente sano. Significa que:

- El primer conjunto era relativamente "amable".

- El nuevo conjunto es más exigente.

- El baseline es funcional pero aún limitado.

Eso es exactamente lo que queremos detectar en un baseline.

---

> *"El que habla la verdad da un testimonio justo, pero el testigo falso dice mentiras. Hay quienes hablan sin pensar y sus palabras hieren como espadas, pero la lengua de los sabios trae sanidad."*
>
> — Proverbios 12:17-18

---

# **Parte 4 — Evaluación cualitativa end-to-end (LLM-as-a-Judge) (Complementaria)**

## **Motivación**

En la **Parte 3** evaluamos formalmente el *baseline de Retrieval* con métricas objetivas (**Recall@k** y **MRR**), aislando el componente generativo. Esto es clave para validar que nuestro motor de recuperación es sólido antes de incorporar la estocasticidad de un LLM.

Sin embargo, en un sistema **RAG**, el usuario final interactúa con la **respuesta generada**, no con los documentos recuperados. Por ello, en esta sección añadimos una **evaluación cualitativa end-to-end** que compare:

- **LLM-only** (sin RAG)  
- **RAG Dense** (solo embeddings)  
- **RAG Hybrid** (BM25 + Dense; combinación ponderada)

y medimos su desempeño con un enfoque de **LLM-as-a-Judge**, donde un segundo LLM actúa como evaluador, asignando puntajes y justificando brevemente la calificación.

### **Importante (Alcance y Disclaimer metodológico)**

- Esta evaluación **NO sustituye** las métricas formales del Retrieval (Parte 3).  
- Sus resultados son **cualitativos** y pueden variar según el modelo y el prompt del juez.  
- Para aumentar reproducibilidad:
  - usaremos **temperatura = 0** tanto en generación como en el juez;
  - el juez evaluará dimensiones separadas (relevancia, fidelidad al contexto, citas, tono pastoral);
  - registraremos también **latencia** y **número de citas**.

El objetivo es presentar una **“demo evaluable”** del sistema, alineada a prácticas comunes en prototipos RAG, y dejar documentadas fortalezas y fallas típicas (p. ej., respuestas convincentes pero no ancladas, o citas irrelevantes).

In [ ]:
# =========================================================
# Parte 4.1 — Setup: OpenAI client, prompts y utilidades
# =========================================================

import os
import time
import textwrap

# ---------------------------
# 1) Validación de prerequisitos
# ---------------------------
required_vars = ["df_chunks", "emb_vbl", "bm25", "st_model"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        f"Faltan variables/objetos requeridos de Partes anteriores: {missing}\n"
        "Asegúrate de haber ejecutado Partes 1–3 antes de la Parte 4."
    )

print("Prerequisitos OK:", required_vars)

# ---------------------------
# 2) OpenAI API Key (Colab/Local)
# ---------------------------
# Tomamos el API Key de OpenAI desde 'secrets' en Colab

from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    print("No se encontró OPENAI_API_KEY en el entorno.")
    print("Antes de ejecutar la Parte 4.2, define la variable de entorno o usa Colab Secrets.")
else:
    print("OPENAI_API_KEY encontrada en el entorno (no se imprime por seguridad).")

# ---------------------------
# 3) Importar OpenAI SDK (nuevo)
# ---------------------------
try:
    from openai import OpenAI
except Exception as e:
    raise ImportError(
        "No se pudo importar 'openai'.\n"
        "Solución típica en Colab:\n"
        "  !pip -q install openai\n"
        "Luego reinicia runtime si es necesario.\n"
        f"Detalle: {e}"
    )

client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

print("Parte 4.1 lista. Siguiente: 4.2 correr LLM-only vs Dense-RAG vs Hybrid-RAG + juez.")

Prerequisitos OK: ['df_chunks', 'emb_vbl', 'bm25', 'st_model']
OPENAI_API_KEY encontrada en el entorno (no se imprime por seguridad).
Parte 4.1 lista. Siguiente: 4.2 correr LLM-only vs Dense-RAG vs Hybrid-RAG + juez.


---

> *“Lo he llenado del Espíritu de Dios, en sabiduría, en inteligencia, en ciencia y en todo arte, para inventar diseños…”*
>
> Éxodo 31:3-5

---

## **4.2 – Evaluación cualitativa con LLM-as-a-Judge (LLM-only vs Dense-RAG vs Hybrid-RAG)**

En esta sección extendemos la evaluación del baseline incorporando el componente generativo del pipeline RAG. A diferencia de la evaluación formal de retrieval (Parte 3), aquí medimos la calidad de la respuesta final bajo tres condiciones:

- **LLM-only**: el modelo responde sin contexto recuperado (riesgo de alucinación).
- **Dense-RAG**: el modelo responde condicionado a pasajes recuperados por embeddings (similitud semántica).
- **Hybrid-RAG**: el modelo responde condicionado a un contexto combinado (léxico + denso), buscando robustez ante variaciones de vocabulario y sinónimos.

Para mantener consistencia, utilizamos un prompt_juez que califica cada respuesta en cuatro dimensiones (1–5):

1. **Relevancia**: qué tan bien responde a la necesidad del usuario.
2. **Faithfulness**: qué tan anclada está la respuesta al contexto provisto (si existe).
3. **Citas**: uso adecuado de referencias bíblicas (precisión y pertinencia).
4. **Tono pastoral**: empatía, cuidado y adecuación al dominio teológico.

Finalmente, calculamos un **score_total** como el promedio de dichas dimensiones y registramos latencia por método. Esta evaluación es cualitativa y no reemplaza métricas cuantitativas, pero ayuda a estimar el impacto del RAG en la experiencia final del usuario y a detectar posibles fallos de anclaje.

In [ ]:
# =========================================
# Parte 4.2 — LLM-only vs Dense-RAG vs Hybrid-RAG + Judge (Salida amigable)
# =========================================

import time
import re
import numpy as np
import pandas as pd

from typing import List, Dict, Optional, Tuple

# --- OpenAI ---
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

# -------------------------
# Helpers: similitud coseno
# -------------------------
def cosine_similarity_matrix(M: np.ndarray, v: np.ndarray) -> np.ndarray:
    """Cosine similarity entre matriz M (n,d) y vector v (d,)."""
    M_norm = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)
    v_norm = v / (np.linalg.norm(v) + 1e-12)
    return (M_norm @ v_norm)

# -------------------------
# Retrieval: BM25 / Dense / Hybrid
# -------------------------
# Requiere que df_chunks tenga columnas: chunk_ref, chunk_text_vbl
TEXT_COL = "chunk_text_vbl"

def bm25_retrieve(query: str, top_k: int = 5) -> List[Dict]:
    scores = bm25.get_scores(query.split())
    top_idx = np.argsort(scores)[::-1][:top_k]
    out = []
    for i in top_idx:
        row = df_chunks.iloc[i]
        out.append({
            "chunk_ref": row["chunk_ref"],
            "chunk_id": row["chunk_id"],
            "score": float(scores[i]),
            "text": row[TEXT_COL],
        })
    return out

def dense_retrieve(query: str, top_k: int = 5, embeddings: np.ndarray = emb_vbl) -> List[Dict]:
    q_emb = st_model.encode([query])[0]
    sims = cosine_similarity_matrix(embeddings, q_emb)
    top_idx = np.argsort(sims)[::-1][:top_k]
    out = []
    for i in top_idx:
        row = df_chunks.iloc[i]
        out.append({
            "chunk_ref": row["chunk_ref"],
            "chunk_id": row["chunk_id"],
            "score": float(sims[i]),
            "text": row[TEXT_COL],
        })
    return out

def hybrid_retrieve(query: str, alpha: float = 0.5, top_k: int = 5) -> List[Dict]:
    """score_h = alpha*bm25_norm + (1-alpha)*dense_norm, luego top-k."""
    # BM25 raw
    bm_scores = bm25.get_scores(query.split())
    bm_scores = np.array(bm_scores, dtype=float)
    bm_norm = (bm_scores - bm_scores.min()) / (bm_scores.max() - bm_scores.min() + 1e-12)

    # Dense raw
    q_emb = st_model.encode([query])[0]
    d_scores = cosine_similarity_matrix(emb_vbl, q_emb)
    d_scores = np.array(d_scores, dtype=float)
    d_norm = (d_scores - d_scores.min()) / (d_scores.max() - d_scores.min() + 1e-12)

    h = alpha * bm_norm + (1 - alpha) * d_norm
    top_idx = np.argsort(h)[::-1][:top_k]

    out = []
    for i in top_idx:
        row = df_chunks.iloc[i]
        out.append({
            "chunk_ref": row["chunk_ref"],
            "chunk_id": row["chunk_id"],
            "score": float(h[i]),
            "bm25_norm": float(bm_norm[i]),
            "dense_norm": float(d_norm[i]),
            "text": row[TEXT_COL],
        })
    return out

# -------------------------
# Construcción de contexto
# -------------------------
def build_context(passages: List[Dict], max_chars: int = 5500) -> str:
    """
    Concatena pasajes como:
    [REF] texto...
    Limita el tamaño para no inflar tokens.
    """
    chunks = []
    total = 0
    for p in passages:
        block = f"[{p['chunk_ref']}] {p['text']}".strip()
        if total + len(block) > max_chars:
            break
        chunks.append(block)
        total += len(block) + 2
    return "\n\n".join(chunks)

# -------------------------
# Generación: respuesta pastoral
# -------------------------
SYSTEM_PASTORAL = """Eres un asistente pastoral cristiano en español.
Tu objetivo es consolar, orientar, despertar sabiduría y discernimiento y animar con fidelidad bíblica.
Reglas:
- Sé empático, claro y cuidadoso.
- Si se provee CONTEXTO, usa SOLO ese contexto para las citas y fundamentos.
- Incluye citas bíblicas explícitas en el formato (LIBRO cap:vers) basadas en el CONTEXTO.
- Evita inventar referencias.
- Estructura sugerida: (1) validación emocional, (2) verdad bíblica, (3) pasos prácticos, (4) oración breve.
"""

def generate_answer(query: str, context_passages: Optional[List[Dict]] = None, model_name: str = "gpt-4o-mini") -> Tuple[str, float]:
    context_text = build_context(context_passages) if context_passages else ""
    user_prompt = f"CONSULTA:\n{query}\n\n"
    if context_text:
        user_prompt += f"CONTEXTO (versículos recuperados):\n{context_text}\n\n"
        user_prompt += "Responde usando SOLAMENTE el CONTEXTO para las citas y fundamentos."
    else:
        user_prompt += "Responde sin contexto externo (no inventes citas específicas si no estás seguro)."

    t0 = time.time()
    resp = client.chat.completions.create(
        model=model_name,
        temperature=0.4,
        messages=[
            {"role": "system", "content": SYSTEM_PASTORAL},
            {"role": "user", "content": user_prompt},
        ],
    )
    latency = time.time() - t0
    return resp.choices[0].message.content.strip(), latency

# -------------------------
# Judge: calificación estructurada
# -------------------------
JUDGE_PROMPT = """Eres un evaluador (judge) estricto. Califica la respuesta a una consulta bíblica/pastoral.
Debes devolver SOLO un JSON válido con estas llaves:
relevancia (1-5), faithfulness (1-5), citas (1-5), tono_pastoral (1-5), comentarios (string breve).
Criterios:
- Relevancia: responde a la necesidad real del usuario.
- Faithfulness: si hay CONTEXTO, no debe agregar afirmaciones/citas fuera del contexto. Si NO hay contexto, evalúa si evita inventar citas específicas.
- Citas: precisión y pertinencia de referencias (si inventa, baja).
- Tono pastoral: empatía, respeto, cuidado (especialmente en crisis).
"""

def count_citations(text: str) -> int:
    # Heurística simple: "Xxx 12:34" o "1 Cor 13:4"
    pattern = r"\b([1-3]\s*)?[A-Za-zÁÉÍÓÚÑáéíóúñ]+\s+\d+:\d+\b"
    return len(re.findall(pattern, text))

def evaluate_with_judge(query: str, context_text: Optional[str], answer: str, model_name: str = "gpt-4o-mini") -> Dict:
    payload = f"QUERY:\n{query}\n\n"
    payload += f"CONTEXTO:\n{context_text if context_text else '(sin contexto)'}\n\n"
    payload += f"RESPUESTA:\n{answer}\n\nDevuelve el JSON."
    t0 = time.time()
    resp = client.chat.completions.create(
        model=model_name,
        temperature=0.0,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": payload},
        ],
        response_format={"type": "json_object"},
    )
    latency = time.time() - t0
    j = resp.choices[0].message.content
    data = pd.read_json(pd.io.common.StringIO(j), typ="series").to_dict()
    data["latency_judge_sec"] = latency
    data["num_citas_detectadas"] = count_citations(answer)
    # Score total (promedio)
    data["score_total"] = np.mean([data["relevancia"], data["faithfulness"], data["citas"], data["tono_pastoral"]])
    return data

# -------------------------
# Pretty print
# -------------------------
def print_block(title: str):
    print("\n" + "="*100)
    print(title)
    print("="*100)

def print_method_result(method_name: str, answer: str, judge: Dict, passages: Optional[List[Dict]] = None, latency_ans: Optional[float] = None):
    print(f"\n--- {method_name} ---")
    if passages:
        print("\n[Contexto recuperado]")
        for i, p in enumerate(passages, 1):
            ref = p["chunk_ref"]
            sc = p["score"]
            extra = ""
            if "bm25_norm" in p and "dense_norm" in p:
                extra = f" | bm25_norm={p['bm25_norm']:.3f} | dense_norm={p['dense_norm']:.3f}"
            print(f"  #{i} | {ref} | score={sc:.4f}{extra}")
    if latency_ans is not None:
        print(f"\n[Latencia respuesta] {latency_ans:.2f} sec")
    print("\n[Respuesta completa]")
    print(answer)

    print("\n[Judge]")
    print(f"  relevancia={judge['relevancia']}/5 | faithfulness={judge['faithfulness']}/5 | citas={judge['citas']}/5 | tono_pastoral={judge['tono_pastoral']}/5")
    print(f"  score_total={judge['score_total']:.2f} | citas_detectadas={judge['num_citas_detectadas']} | lat_judge={judge['latency_judge_sec']:.2f}s")
    print(f"  comentarios: {judge['comentarios']}")

# -------------------------
# Ejecutar experimento
# -------------------------
EVAL_QUERIES = [
    "Siento que fracasé en la vida. He considerado suicidarme. ¿Qué me diría Dios ahora?",
    "Vivo en constante frustración, me siento deprimido y mi corazón está sumamente entristecido. ¿Qué puedo hacer?",
    "¿Qué dice la Biblia sobre la tecnología y la ciencia?",
    "Tengo miedo del futuro y siento ansiedad constante. ¿Cómo puedo hallar paz?",
    "Estoy cargando demasiadas responsabilidades y me siento incapaz. ¿Qué dice la Biblia?",
]

TOP_K = 5
ALPHA = 0.5

all_rows = []

for q in EVAL_QUERIES:
    print_block(f"QUERY: {q}")

    # --- LLM-only ---
    ans_llm, lat_llm = generate_answer(q, context_passages=None)
    judge_llm = evaluate_with_judge(q, None, ans_llm)
    print_method_result("LLM-only", ans_llm, judge_llm, passages=None, latency_ans=lat_llm)

    # --- Dense-RAG ---
    dense_pass = dense_retrieve(q, top_k=TOP_K)
    ctx_dense = build_context(dense_pass)
    ans_dense, lat_dense = generate_answer(q, context_passages=dense_pass)
    judge_dense = evaluate_with_judge(q, ctx_dense, ans_dense)
    print_method_result("Dense-RAG", ans_dense, judge_dense, passages=dense_pass, latency_ans=lat_dense)

    # --- Hybrid-RAG ---
    hyb_pass = hybrid_retrieve(q, alpha=ALPHA, top_k=TOP_K)
    ctx_hyb = build_context(hyb_pass)
    ans_hyb, lat_hyb = generate_answer(q, context_passages=hyb_pass)
    judge_hyb = evaluate_with_judge(q, ctx_hyb, ans_hyb)
    print_method_result(f"Hybrid-RAG (α={ALPHA})", ans_hyb, judge_hyb, passages=hyb_pass, latency_ans=lat_hyb)

    # --- Guardar en tabla resumen ---
    for sys_name, ans, lat, judge in [
        ("LLM-only", ans_llm, lat_llm, judge_llm),
        ("Dense-RAG", ans_dense, lat_dense, judge_dense),
        (f"Hybrid-RAG α={ALPHA}", ans_hyb, lat_hyb, judge_hyb),
    ]:
        all_rows.append({
            "query": q,
            "system": sys_name,
            "relevancia": judge["relevancia"],
            "faithfulness": judge["faithfulness"],
            "citas": judge["citas"],
            "tono_pastoral": judge["tono_pastoral"],
            "score_total": round(judge["score_total"], 2),
            "latency_sec": round(lat, 2),
            "num_citas_detectadas": judge["num_citas_detectadas"],
            "comentarios": judge["comentarios"],
            "answer_full": ans,  # por si luego quieres exportar
        })

df_judge = pd.DataFrame(all_rows)
print_block("Resumen tabular (df_judge)")
display(df_judge.drop(columns=["answer_full"]))


QUERY: Siento que fracasé en la vida. He considerado suicidarme. ¿Qué me diría Dios ahora?

--- LLM-only ---

[Latencia respuesta] 6.44 sec

[Respuesta completa]
Lamento profundamente que te sientas así. Es muy doloroso sentirse atrapado y pensar que no hay salida. Quiero que sepas que tus sentimientos son válidos y que no estás solo en esto. Dios se preocupa profundamente por ti y por tu bienestar.

La verdad bíblica nos recuerda que, incluso en los momentos más oscuros, Dios está presente. En Salmos 34:18 se nos dice: "Cercano está Jehová a los quebrantados de corazón; y salva a los contritos de espíritu." Esto nos muestra que Dios está cerca de aquellos que están sufriendo y que Él se preocupa por cada una de nuestras luchas.

Aquí hay algunos pasos prácticos que podrías considerar: 
1. Habla con alguien de confianza sobre cómo te sientes. Puede ser un amigo, un familiar o un consejero.
2. Busca apoyo en tu comunidad de fe. Ellos pueden ofrecerte amor y ayuda en este momento difíci

,query,system,relevancia,faithfulness,citas,tono_pastoral,score_total,latency_sec,num_citas_detectadas,comentarios
0,Siento que fracasé en la vida. He considerado suicidarme. ¿Qué me diría Dios ahora?,LLM-only,5,4,5,5,4.75,6.44,1,"La respuesta es muy relevante y empática, abordando la crisis del usuario con cuidado. La cita de Salmos es pertinente y se utiliza adecuadamente en el contexto de sufrimiento. Sin embargo, se podría mejorar la faithfulness al evitar la afirmación de que 'la verdad bíblica nos recuerda', ya que ..."
1,Siento que fracasé en la vida. He considerado suicidarme. ¿Qué me diría Dios ahora?,Dense-RAG,5,5,5,5,5.00,10.27,3,"La respuesta aborda adecuadamente la angustia del usuario, valida sus sentimientos y ofrece consuelo y esperanza a través de citas bíblicas pertinentes, manteniendo un tono empático y pastoral."
2,Siento que fracasé en la vida. He considerado suicidarme. ¿Qué me diría Dios ahora?,Hybrid-RAG α=0.5,5,5,5,5,5.00,12.63,2,"La respuesta aborda la angustia del usuario con empatía y ofrece consuelo y pasos prácticos, utilizando citas bíblicas pertinentes y en contexto."
3,"Vivo en constante frustración, me siento deprimido y mi corazón está sumamente entristecido. ¿Qué puedo hacer?",LLM-only,5,4,5,5,4.75,6.46,1,"La respuesta es empática y ofrece pasos prácticos, aunque la cita de Salmos es adecuada, se podría haber mencionado más sobre el contexto de la tristeza."
4,"Vivo en constante frustración, me siento deprimido y mi corazón está sumamente entristecido. ¿Qué puedo hacer?",Dense-RAG,5,5,5,5,5.00,9.90,4,"La respuesta aborda adecuadamente la angustia del usuario, utiliza citas relevantes y ofrece un tono empático y pastoral."
5,"Vivo en constante frustración, me siento deprimido y mi corazón está sumamente entristecido. ¿Qué puedo hacer?",Hybrid-RAG α=0.5,5,5,5,5,5.00,12.21,3,"La respuesta aborda adecuadamente la angustia del usuario, utiliza citas relevantes y ofrece un tono empático y de apoyo."
6,¿Qué dice la Biblia sobre la tecnología y la ciencia?,LLM-only,5,4,4,5,4.50,7.24,1,"La respuesta es relevante y aborda la curiosidad del usuario sobre tecnología y ciencia desde una perspectiva bíblica, aunque la cita de Proverbios podría ser más contextualizada."
7,¿Qué dice la Biblia sobre la tecnología y la ciencia?,Dense-RAG,5,5,5,5,5.00,8.06,2,"La respuesta aborda adecuadamente la inquietud sobre la relación entre tecnología, ciencia y fe, utilizando citas bíblicas pertinentes y ofreciendo un tono pastoral empático."
8,¿Qué dice la Biblia sobre la tecnología y la ciencia?,Hybrid-RAG α=0.5,5,5,5,5,5.00,6.26,3,"La respuesta aborda adecuadamente la consulta sobre tecnología y ciencia, utilizando citas bíblicas pertinentes y ofreciendo un tono empático y práctico."
9,Tengo miedo del futuro y siento ansiedad constante. ¿Cómo puedo hallar paz?,LLM-only,5,5,5,5,5.00,7.92,1,"La respuesta aborda adecuadamente la ansiedad del usuario, ofrece consuelo bíblico y pasos prácticos, manteniendo un tono empático y pastoral."


### **Interpretación de resultados (Parte 4.2 — LLM-only vs Dense-RAG vs Hybrid-RAG + Judge)**

Los resultados muestran que el *Judge* asigna puntajes altos (casi siempre 4.5–5.0) a los tres sistemas evaluados (LLM-only, Dense-RAG y Hybrid-RAG). Esto sugiere que, con los criterios actuales del juez, las diferencias entre métodos no se reflejan fuertemente en la métrica final; sin embargo, el análisis cualitativo sí revela señales importantes:

1. **El RAG aporta anclaje bíblico verificable.**  
   En consultas emocionalmente delicadas (p.ej., desesperanza o ideación suicida), Dense-RAG e Hybrid-RAG tienden a recuperar pasajes directamente relacionados con sufrimiento, angustia y clamor (Salmos, Job, Lamentaciones). Esto permite respuestas más “aterrizadas” y con mayor trazabilidad hacia el texto bíblico recuperado.

2. **LLM-only mantiene buen tono, pero es más propenso a “citas genéricas” o menos contextualizadas.**  
   En algunos casos, el Judge detecta que la respuesta LLM-only usa versículos típicos (p.ej., Jeremías 29:11) que pueden ser correctos en sentido amplio, pero no necesariamente los más pertinentes para la situación inmediata del usuario. Esto se refleja en pequeñas caídas en *faithfulness* y *citas*.

3. **Hybrid-RAG puede introducir ruido léxico.**  
   Al combinar BM25 + embeddings, Hybrid-RAG ocasionalmente recupera pasajes con buena coincidencia literal (keywords), pero menos directos para el objetivo pastoral/práctico. Esto no siempre es castigado por el Judge actual.

4. **Limitación clave: el Judge es poco discriminativo y demasiado generoso.**  
   Dado que la mayoría de respuestas reciben 5/5, se vuelve difícil usar el score como señal fina para comparar métodos. Además, el Judge no penaliza fuertemente cuando la respuesta menciona referencias bíblicas que **no están** en el contexto recuperado (posible “citation leakage”).

**Conclusión:** Aunque el score total cambia poco, el experimento confirma que RAG (especialmente Dense-RAG) ayuda a mejorar la trazabilidad y alineación temática con pasajes recuperados. Para obtener una evaluación más realista, se requiere endurecer el Judge, penalizar citas no soportadas por el contexto y, si es necesario, forzar una reescritura “context-faithful”.

---

> *“Y dediqué mi corazón a inquirir y a explorar con sabiduría todo lo que se hace bajo el cielo…”*
>
> Eclesiastés 1:13 –

---

## **4.2.1 Endureciendo la evaluación: detección de citas fuera del contexto y penalización automática**

En la Parte 4.2 observamos puntajes muy altos y poca discriminación entre métodos. Una causa frecuente es que el Judge puede premiar tono/estructura incluso cuando la respuesta incluye referencias bíblicas no respaldadas explícitamente por el contexto recuperado (p. ej., el modelo “recuerda” Filipenses o Jeremías sin estar en los pasajes entregados).

Para hacer la evaluación más estricta y trazable, implementamos:

1. **Extractor de citas**: detecta referencias tipo `PSA 34:18`, `MAT 11:28-30`, `Jeremías 29:11`, etc.
2. **Verificación contra contexto**: compara las citas mencionadas en la respuesta con el conjunto de pasajes realmente recuperados.
3. **Penalización automática**: si hay citas fuera de contexto, reducimos las métricas de *citas* y *faithfulness* y añadimos un comentario explícito.
4. **(Opcional) Repair pass**: si se detectan citas fuera de contexto, pedimos al modelo reescribir la respuesta usando **solo** los pasajes disponibles, para reducir “citation leakage”.

Este cambio busca que la métrica refleje mejor la fidelidad al contexto recuperado, y que el score sea más útil para comparar LLM-only vs RAG.

In [ ]:
import re
import pandas as pd
from typing import List, Dict, Tuple, Optional

# =========================
# 1) Extractor de citas
# =========================

# Detecta formatos tipo "PSA 34:18", "MAT 11:28-30", "1CO 1:31", "2TI 3:16-17"
REF_PATTERN = re.compile(
    r"\b(?:[1-3]\s*)?[A-Z]{3}\s+\d{1,3}:\d{1,3}(?:-\d{1,3})?\b"
)

SPANISH_BOOK_PATTERN = re.compile(
    r"\b[A-ZÁÉÍÓÚÑa-záéíóúñ]+(?:\s+[A-ZÁÉÍÓÚÑa-záéíóúñ]+)*\s+\d{1,3}:\d{1,3}(?:-\d{1,3})?\b"
)

def extract_chunk_refs(text: str) -> List[str]:
    """Extrae refs canónicas tipo 'PSA 34:18'."""
    if not text:
        return []
    refs = REF_PATTERN.findall(text.upper())
    # Normaliza espacios múltiples
    refs = [re.sub(r"\s+", " ", r).strip() for r in refs]
    # Dedup preservando orden
    seen = set()
    out = []
    for r in refs:
        if r not in seen:
            seen.add(r)
            out.append(r)
    return out

def extract_context_refs(context_passages: Optional[List[Dict]]) -> List[str]:
    """
    Espera context_passages como lista de dicts con al menos {"chunk_ref": "..."}.
    Devuelve lista de chunk_ref normalizados.
    """
    if not context_passages:
        return []
    refs = []
    for p in context_passages:
        if isinstance(p, dict) and "chunk_ref" in p:
            refs.append(str(p["chunk_ref"]).upper().strip())
    # Dedup
    seen = set()
    out = []
    for r in refs:
        if r not in seen:
            seen.add(r)
            out.append(r)
    return out


# =========================
# 2) Penalización estricta
# =========================

def apply_strict_citation_penalty(
    judge_dict: Dict,
    refs_in_answer: List[str],
    refs_in_context: List[str],
    penalty_faith: int = 2,
    penalty_citas: int = 2
) -> Dict:
    """
    Si la respuesta menciona citas (chunk_ref) que NO están en el contexto, penaliza.
    - penalty_faith, penalty_citas: cuánto bajar (con piso 1).
    """
    jd = dict(judge_dict) if judge_dict else {}
    missing = [r for r in refs_in_answer if r not in set(refs_in_context)]

    jd["refs_en_respuesta"] = refs_in_answer
    jd["refs_en_contexto"] = refs_in_context
    jd["refs_fuera_contexto"] = missing
    jd["num_refs_fuera_contexto"] = len(missing)

    # Solo penalizamos si hay contexto (si no hay contexto, LLM-only puede citar "libremente")
    has_context = len(refs_in_context) > 0

    if has_context and len(missing) > 0:
        # Baja faithfulness y citas
        if "faithfulness" in jd and isinstance(jd["faithfulness"], (int, float)):
            jd["faithfulness"] = max(1, int(round(jd["faithfulness"])) - penalty_faith)
        if "citas" in jd and isinstance(jd["citas"], (int, float)):
            jd["citas"] = max(1, int(round(jd["citas"])) - penalty_citas)

        # Recalcula score_total si existe y es promedio simple (asumimos 4 métricas)
        # Si el score_total se calcula distinto, comentar esta parte.
        if all(k in jd for k in ["relevancia", "faithfulness", "citas", "tono_pastoral"]):
            try:
                vals = [float(jd["relevancia"]), float(jd["faithfulness"]), float(jd["citas"]), float(jd["tono_pastoral"])]
                jd["score_total"] = sum(vals) / len(vals)
            except Exception:
                pass

        # Anexa comentario
        extra = f"Penalización: se detectaron {len(missing)} refs fuera de contexto: {missing}."
        if "comentarios" in jd and jd["comentarios"]:
            jd["comentarios"] = str(jd["comentarios"]).rstrip() + " " + extra
        else:
            jd["comentarios"] = extra

    return jd


# =========================
# 3) Repair pass
# =========================

def repair_answer_if_needed(
    query: str,
    answer: str,
    context_passages: List[Dict],
    refs_missing: List[str],
    client_generate_fn
) -> Tuple[str, bool]:
    """
    Si hay refs fuera de contexto, reescribe la respuesta con una regla estricta.
    client_generate_fn: función que recibe (query, context_passages) y devuelve (answer_text, latency).
    """
    if not context_passages or not refs_missing:
        return answer, False

    # Forzamos reescritura: usar solo contexto, NO inventar refs
    strict_query = (
        "REESCRIBE la respuesta pastoral cumpliendo estas reglas:\n"
        "1) Usa únicamente la información y pasajes del CONTEXTO proporcionado.\n"
        "2) No cites referencias bíblicas que NO estén en el contexto.\n"
        "3) Mantén estructura: (1) validación emocional, (2) verdad bíblica, (3) pasos prácticos, (4) oración.\n\n"
        f"Pregunta del usuario: {query}\n"
        f"Respuesta previa (para mejorar): {answer}\n"
    )

    repaired, _lat = client_generate_fn(strict_query, context_passages=context_passages)
    return repaired, True


# =========================
# 4) Evaluación estricta wrapper
# =========================

def run_strict_judge_for_triplet(
    query: str,
    ctx_dense: Optional[List[Dict]],
    ctx_hybrid: Optional[List[Dict]],
    generate_answer_fn,
    evaluate_judge_fn,
    do_repair: bool = True
) -> pd.DataFrame:
    """
    Corre LLM-only, Dense-RAG, Hybrid-RAG y aplica:
    - extracción de refs
    - penalización estricta por refs fuera de contexto (ya existente)
    - (nuevo) métricas GCF1 de groundedness de citas
    - repair pass opcional + re-evaluación
    """

    rows = []

    def _eval_one(system_name: str, context_passages: Optional[List[Dict]]):
        ans, lat = generate_answer_fn(query, context_passages=context_passages)
        judge = evaluate_judge_fn(query, context_passages, ans)

        refs_ans = extract_chunk_refs(ans)
        refs_ctx = extract_context_refs(context_passages)
        judge_strict = apply_strict_citation_penalty(judge, refs_ans, refs_ctx)

        gcf1 = grounded_citation_metrics(ans, context_passages)

        repaired = False
        if do_repair and judge_strict.get("num_refs_fuera_contexto", 0) > 0:
            ans2, repaired = repair_answer_if_needed(
                query=query,
                answer=ans,
                context_passages=context_passages,
                refs_missing=judge_strict.get("refs_fuera_contexto", []),
                client_generate_fn=generate_answer_fn
            )
            if repaired:
                judge2 = evaluate_judge_fn(query, context_passages, ans2)
                refs_ans2 = extract_chunk_refs(ans2)
                # Recalcula penalización con la respuesta reparada
                judge_strict = apply_strict_citation_penalty(judge2, refs_ans2, refs_ctx)
                # Recalcula GCF1 con respuesta reparada
                gcf1 = grounded_citation_metrics(ans2, context_passages)
                ans = ans2

        row = {
            "query": query,
            "system": system_name + (" (repaired)" if repaired else ""),
            "latency_sec": lat,
            "answer_full": ans,
            **judge_strict,
            **gcf1,  # <-- agrega grounded_prec, grounded_rec, grounded_f1, etc.
        }
        rows.append(row)

    _eval_one("LLM-only", None)  # GCF1=NaN por diseño
    _eval_one("Dense-RAG", ctx_dense)
    _eval_one("Hybrid-RAG", ctx_hybrid)

    return pd.DataFrame(rows)


# =========================
# 5) Ejecutar experimento (estricto) sobre las queries
# =========================

# IMPORTANTE:
# Se asume algo como:
# - dense_retrieve(query, top_k=5) -> lista de dicts con chunk_ref, text, score
# - hybrid_retrieve(query, alpha=0.5, top_k=5) -> lista similar
#
# Y que generate_answer(query, context_passages=None) usa context_passages cuando no es None.

STRICT_TOP_K = 5
STRICT_ALPHA = 0.5
DO_REPAIR = True

eval_queries = [
    "Siento que fracasé en la vida. He considerado suicidarme. ¿Qué me diría Dios ahora?",
    "Vivo en constante frustración, me siento deprimido y mi corazón está sumamente entristecido. ¿Qué puedo hacer?",
    "¿Qué dice la Biblia sobre la tecnología y la ciencia?",
    "Tengo miedo del futuro y siento ansiedad constante. ¿Cómo puedo hallar paz?",
    "Estoy cargando demasiadas responsabilidades y me siento incapaz. ¿Qué dice la Biblia?"
]

all_strict = []

for q in eval_queries:
    ctx_dense = dense_retrieve(q, top_k=STRICT_TOP_K)
    ctx_hybrid = hybrid_retrieve(q, alpha=STRICT_ALPHA, top_k=STRICT_TOP_K)

    df_one = run_strict_judge_for_triplet(
        query=q,
        ctx_dense=ctx_dense,
        ctx_hybrid=ctx_hybrid,
        generate_answer_fn=generate_answer,
        evaluate_judge_fn=evaluate_with_judge,
        do_repair=DO_REPAIR
    )
    all_strict.append(df_one)

df_strict = pd.concat(all_strict, ignore_index=True)

# Resumen amigable (sin truncar respuestas aún)
summary_cols = [
    "query", "system",
    "relevancia", "faithfulness", "citas", "tono_pastoral",
    "score_total",
    "num_refs_fuera_contexto",
    "refs_fuera_contexto",
    "latency_sec",
    "comentarios"
]
df_strict_summary = df_strict[summary_cols].copy()

df_strict_summary.sort_values(["query", "system"], inplace=True)
df_strict_summary.reset_index(drop=True, inplace=True)

df_strict_summary

,query,system,relevancia,faithfulness,citas,tono_pastoral,score_total,num_refs_fuera_contexto,refs_fuera_contexto,latency_sec,comentarios
0,Estoy cargando demasiadas responsabilidades y me siento incapaz. ¿Qué dice la Biblia?,Dense-RAG,5,5,5,5,5.00,0,[],6.280471,"La respuesta aborda adecuadamente la necesidad del usuario, proporciona un contexto bíblico relevante y mantiene un tono empático y pastoral."
1,Estoy cargando demasiadas responsabilidades y me siento incapaz. ¿Qué dice la Biblia?,Hybrid-RAG,5,5,5,5,5.00,0,[],9.480873,"La respuesta aborda adecuadamente la necesidad del usuario, utiliza citas bíblicas pertinentes y mantiene un tono empático y pastoral."
2,Estoy cargando demasiadas responsabilidades y me siento incapaz. ¿Qué dice la Biblia?,LLM-only,5,5,5,5,5.00,0,[],5.682644,"La respuesta aborda adecuadamente la necesidad del usuario, ofrece un versículo relevante y práctico, y mantiene un tono empático y pastoral."
3,Siento que fracasé en la vida. He considerado suicidarme. ¿Qué me diría Dios ahora?,Dense-RAG (repaired),5,3,3,5,4.00,3,"[LAM 3:17, JOB 6:10, PSA 116:4]",11.956130,"La respuesta aborda adecuadamente la angustia del usuario, utiliza citas bíblicas pertinentes y mantiene un tono empático y de apoyo. Penalización: se detectaron 3 refs fuera de contexto: ['LAM 3:17', 'JOB 6:10', 'PSA 116:4']."
4,Siento que fracasé en la vida. He considerado suicidarme. ¿Qué me diría Dios ahora?,Hybrid-RAG,5,5,5,5,5.00,0,[],9.063278,"La respuesta aborda adecuadamente la angustia del usuario, utiliza citas bíblicas pertinentes y mantiene un tono empático y de apoyo."
5,Siento que fracasé en la vida. He considerado suicidarme. ¿Qué me diría Dios ahora?,LLM-only,5,4,5,5,4.75,0,[],6.399877,"La respuesta es empática y ofrece esperanza, aunque la cita de Jeremías 29:11 podría ser interpretada de manera más contextualizada."
6,Tengo miedo del futuro y siento ansiedad constante. ¿Cómo puedo hallar paz?,Dense-RAG,5,5,5,5,5.00,0,[],10.364495,"La respuesta aborda adecuadamente la ansiedad del usuario, utiliza citas relevantes y ofrece un tono empático y pastoral."
7,Tengo miedo del futuro y siento ansiedad constante. ¿Cómo puedo hallar paz?,Hybrid-RAG,5,5,5,5,5.00,0,[],11.346330,"La respuesta es empática y se basa en el contexto bíblico, ofreciendo consuelo y pasos prácticos para hallar paz."
8,Tengo miedo del futuro y siento ansiedad constante. ¿Cómo puedo hallar paz?,LLM-only,5,5,5,5,5.00,0,[],11.264396,"La respuesta aborda adecuadamente la ansiedad del usuario, ofrece consuelo bíblico y pasos prácticos, manteniendo un tono empático y pastoral."
9,"Vivo en constante frustración, me siento deprimido y mi corazón está sumamente entristecido. ¿Qué puedo hacer?",Dense-RAG (repaired),5,3,3,5,4.00,1,[JOB 30:28],9.147594,"La respuesta es muy relevante y empática, utilizando citas bíblicas adecuadas y en contexto, ofreciendo pasos prácticos y una oración pastoral. Penalización: se detectaron 1 refs fuera de contexto: ['JOB 30:28']."


### **Interpretación de resultados – 4.2.1 Judge estricto y penalización por citas fuera de contexto**

La implementación del Judge estricto introdujo dos mejoras metodológicas importantes:

1. Detección automática de referencias bíblicas mencionadas en la respuesta.
2. Penalización cuando dichas referencias no estaban presentes en el contexto recuperado (citation leakage).
3. Repair pass opcional para forzar reescritura basada exclusivamente en el contexto.

#### **Hallazgos clave:**

1. **La mayoría de respuestas siguen obteniendo score perfecto (5.0).**  
   Esto indica que:
   - El sistema (especialmente Dense-RAG) está generando respuestas sólidas.
   - El Judge sigue siendo relativamente permisivo.
   - Las respuestas tienden a ser coherentes y pastoralmente apropiadas.

2. **El Judge estricto sí logró detectar leakage en algunos casos.**  
   Ejemplos:
   - Hybrid-RAG en la consulta sobre suicidio fue penalizado por citar LAM 3:17 y PSA 17:3 fuera del contexto.
   - Dense y Hybrid en el caso de frustración/depresión fueron penalizados por múltiples citas no recuperadas.

   Esto confirma que:
   - El modelo tiende a "recordar" versículos adicionales aunque no estén en el contexto.
   - La penalización hace que la métrica sea más informativa.

3. **Dense-RAG se mantiene como el sistema más estable.**  
   En la mayoría de consultas:
   - No presentó citas fuera de contexto.
   - Conservó score perfecto.
   - Mostró mayor consistencia que Hybrid.

4. **LLM-only no es castigado en citation leakage porque no depende de contexto.**  
   Esto revela una diferencia conceptual:
   - LLM-only es libre de citar.
   - RAG debe demostrar fidelidad al contexto recuperado.
   Por lo tanto, el estándar de evaluación para RAG es más estricto (como debe ser).

#### **Conclusión metodológica:**

El Judge estricto mejora la evaluación, pero el sistema sigue mostrando:

- Alta calidad pastoral.
- Buena relevancia.
- Diferencias más sutiles entre métodos.

Esto sugiere que:
- Dense-RAG ofrece el mejor equilibrio entre fidelidad y coherencia.
- Hybrid-RAG introduce ocasionalmente ruido léxico.
- LLM-only puede ser sólido, pero carece de trazabilidad verificable.

Con esto, el Avance 3 queda metodológicamente robusto y comparativamente sólido.

---

> *“Porque las cosas invisibles de él, su eterno poder y deidad, se hacen claramente visibles desde la creación del mundo, siendo entendidas por medio de las cosas hechas…”*  
>
> Romanos 1:20

---

## **4.2.2 Medición de qué tan bien la respuesta generada usa el contexto recuperado con la métrica Grounded Citation F1 (GCF1)**

Cuando el sistema cita versículos, ¿están respaldados por el contexto recuperado? ¿Y qué tanto del contexto se aprovechó en la respuesta?, para responder a estas preguntas utilizaremos una nueva métrica que cuantifica de manera simple y robusta la fidelidad citacional y el aprovechamiento del contexto en una única cifra comparable entre sistemas.

In [ ]:
def grounded_citation_metrics(answer_text, context_passages):
    """
    Calcula Precision, Recall y F1 de citas 'ancladas' al contexto recuperado.
    - answer_text: texto completo de la respuesta
    - context_passages: lista de dicts con al menos {"chunk_ref": "..."}
    Devuelve: dict con grounded_prec, grounded_rec, grounded_f1,
              num_refs_answer, num_refs_context, num_refs_grounded,
              has_grounded_cite (0/1)
    """
    refs_ans = extract_chunk_refs(answer_text)
    refs_ctx = extract_context_refs(context_passages)

    set_ans = set(refs_ans)
    set_ctx = set(refs_ctx)

    if len(set_ans) == 0:
        grounded_prec = np.nan
    else:
        grounded_prec = len(set_ans & set_ctx) / len(set_ans)

    if len(set_ctx) == 0:
        grounded_rec = np.nan
    else:
        grounded_rec = len(set_ans & set_ctx) / len(set_ctx)

    if np.isnan(grounded_prec) or np.isnan(grounded_rec) or (grounded_prec + grounded_rec) == 0:
        grounded_f1 = np.nan
    else:
        grounded_f1 = 2 * grounded_prec * grounded_rec / (grounded_prec + grounded_rec)

    return {
        "grounded_prec": grounded_prec,
        "grounded_rec": grounded_rec,
        "grounded_f1": grounded_f1,
        "num_refs_answer": len(set_ans),
        "num_refs_context": len(set_ctx),
        "num_refs_grounded": len(set_ans & set_ctx),
        "has_grounded_cite": int(len(set_ans & set_ctx) > 0),
        "context_refs_list": list(refs_ctx),
        "answer_refs_list": list(refs_ans),
    }

In [ ]:
gcf1_cols = ["grounded_prec", "grounded_rec", "grounded_f1", "has_grounded_cite"]
judge_cols = ["relevancia", "faithfulness", "citas", "tono_pastoral", "score_total", "latency_sec"]

df_gcf1_summary = (
    df_strict
    .groupby("system")[gcf1_cols + judge_cols + ["num_refs_fuera_contexto"]]
    .mean(numeric_only=True)
    .reset_index()
    .rename(columns={"num_refs_fuera_contexto": "leakage_avg"})
)

df_ranked = (
    df_gcf1_summary
    .assign(
        composite = 0.6*df_gcf1_summary["score_total"] + 0.4*df_gcf1_summary["grounded_f1"].fillna(0.0)
    )
    .sort_values(by=["composite", "grounded_f1", "score_total", "leakage_avg"], ascending=[False, False, False, True])
)

print("Resumen con GCF1 por sistema:")
display(df_gcf1_summary.sort_values("grounded_f1", ascending=False))

print("Ranking sugerido (composite):")
display(df_ranked)

Resumen con GCF1 por sistema:


,system,grounded_prec,grounded_rec,grounded_f1,has_grounded_cite,relevancia,faithfulness,citas,tono_pastoral,score_total,latency_sec,leakage_avg
3,Hybrid-RAG (repaired),0.80,0.80,0.800000,1.000000,5.0,3.0,3.0,5.0,4.0,9.541876,1.0
0,Dense-RAG,1.00,0.20,0.750000,0.333333,5.0,5.0,5.0,5.0,5.0,8.290305,0.0
2,Hybrid-RAG,1.00,0.25,0.611111,0.500000,5.0,5.0,5.0,5.0,5.0,9.511540,0.0
1,Dense-RAG (repaired),0.25,0.10,0.285714,0.500000,5.0,3.0,3.0,5.0,4.0,10.551862,2.0
4,LLM-only,NaN,NaN,NaN,0.000000,5.0,4.4,4.8,5.0,4.8,7.160938,0.0


Ranking sugerido (composite):


,system,grounded_prec,grounded_rec,grounded_f1,has_grounded_cite,relevancia,faithfulness,citas,tono_pastoral,score_total,latency_sec,leakage_avg,composite
0,Dense-RAG,1.00,0.20,0.750000,0.333333,5.0,5.0,5.0,5.0,5.0,8.290305,0.0,3.300000
2,Hybrid-RAG,1.00,0.25,0.611111,0.500000,5.0,5.0,5.0,5.0,5.0,9.511540,0.0,3.244444
4,LLM-only,NaN,NaN,NaN,0.000000,5.0,4.4,4.8,5.0,4.8,7.160938,0.0,2.880000
3,Hybrid-RAG (repaired),0.80,0.80,0.800000,1.000000,5.0,3.0,3.0,5.0,4.0,9.541876,1.0,2.720000
1,Dense-RAG (repaired),0.25,0.10,0.285714,0.500000,5.0,3.0,3.0,5.0,4.0,10.551862,2.0,2.514286


## **Interpretación de resultados**

* Dense‑RAG (3.30)
* Hybrid‑RAG (3.24)
* LLM‑only (2.88)
* Hybrid‑RAG (repaired) (2.72)
* Dense‑RAG (repaired) (2.51)

Cuando ponderamos calidad global del juez (60%) y uso del contexto (40%), **Dense‑RAG** resulta **ligeramente superior** a Hybrid‑RAG, y ambos superan a LLM‑only debido a su trazabilidad (aunque LLM‑only tenga alta puntuación del juez). Los modos “repaired” no aportan ventajas sostenidas; mejoran a veces el uso del contexto, pero penalizan naturalidad y consistencia, e incluso elevan leakage bajo el esquema estricto.

## **4.3 Análisis comparativo final: LLM-only vs Dense-RAG vs Hybrid-RAG**

Después de:

* Evaluar Retrieval (Recall, MRR).
* Implementar Judge automático (relevancia, faithfulness, citas, tono).
* Endurecer evaluación con penalización por citas fuera de contexto.
* Ejecutar repair pass.
* Introducir la métrica Grounded Citation F1 (GCF1) para medir uso del contexto.

Procedemos a un análisis consolidado.

**Objetivos**

* Comparar desempeño promedio por sistema.
* Analizar estabilidad entre consultas.
* Evaluar el impacto real del RAG frente a LLM‑only.
* Determinar la arquitectura más robusta para aplicaciones pastorales basadas en Biblia.

In [ ]:
# ==========================================
# 4.3.1 Promedios por sistema
# ==========================================

metrics_cols = [
    "relevancia",
    "faithfulness",
    "citas",
    "tono_pastoral",
    "score_total",
    "num_refs_fuera_contexto",
    "latency_sec"
]

df_summary = (
    df_strict
    .groupby("system")[metrics_cols]
    .mean()
    .reset_index()
)

df_summary.sort_values("score_total", ascending=False)

,system,relevancia,faithfulness,citas,tono_pastoral,score_total,num_refs_fuera_contexto,latency_sec
0,Dense-RAG,5.0,5.0,5.0,5.0,5.00,0.0,10.081056
2,Hybrid-RAG,5.0,5.0,5.0,5.0,5.00,0.0,10.467595
4,LLM-only,5.0,4.6,4.8,5.0,4.85,0.0,7.074723
1,Dense-RAG (repaired),5.0,3.0,3.0,5.0,4.00,2.0,11.747798
3,Hybrid-RAG (repaired),5.0,3.0,3.0,5.0,4.00,4.0,8.954060


In [ ]:
# ==========================================
# 4.3.3 Conteo total de citas fuera de contexto
# ==========================================

df_leakage = (
    df_strict
    .groupby("system")["num_refs_fuera_contexto"]
    .sum()
    .reset_index()
)

df_leakage.sort_values("num_refs_fuera_contexto")

,system,num_refs_fuera_contexto
0,Dense-RAG,0
2,Hybrid-RAG,0
4,LLM-only,0
1,Dense-RAG (repaired),6
3,Hybrid-RAG (repaired),8


In [ ]:
# ==========================================
# 4.3.2 Varianza por sistema
# ==========================================

df_variance = (
    df_strict
    .groupby("system")[metrics_cols]
    .std()
    .reset_index()
)

df_variance

,system,relevancia,faithfulness,citas,tono_pastoral,score_total,num_refs_fuera_contexto,latency_sec
0,Dense-RAG,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.627203
1,Dense-RAG (repaired),0.0,0.000000,0.000000,0.0,0.000000,1.732051,4.442938
2,Hybrid-RAG,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.748268
3,Hybrid-RAG (repaired),0.0,0.000000,0.000000,0.0,0.000000,0.000000,1.534948
4,LLM-only,0.0,0.547723,0.447214,0.0,0.136931,0.000000,1.243492


---

> *"Por lo tanto, ya que estamos recibiendo un reino que no puede ser sacudido, seamos agradecidos. Así podremos servir a Dios de una manera que le agrade, con respeto y temor reverente."*
>
> Hebreos 12:28

---

# **Conclusión general del Avance 4**

En esta fase se evaluaron tres arquitecturas principales:

1. LLM-only
2. Dense-RAG
3. Hybrid-RAG (BM25 + Dense)

Además, se evaluó una versión “repaired” cuando el sistema fue forzado a reescribir bajo restricciones estrictas de contexto.

---

### *1. Resultados agregados (sin repair)*

Promedios principales observados:


**Dense‑RAG:**

* Judge: score_total = 5.0
* Groundedness: GCF1 = 0.75 (precisión = 1.00; recall = 0.20)
* Leakage: 0.0
* Latencia: ~8.29 s



**Hybrid‑RAG:**

* Judge: score_total = 5.0
* Groundedness: GCF1 ≈ 0.61 (precisión = 1.00; recall = 0.25)
* Leakage: 0.0
* Latencia: ~9.51 s



**LLM‑only:**

* Judge: score_total ≈ 4.8
* Groundedness: no aplica (NaN)
* Latencia: ~7.16 s



**Interpretación:**

* Dense y Hybrid logran calidad máxima según el juez y cero fuga de citas.
* Dense‑RAG obtiene mejor GCF1 que Hybrid‑RAG en este conjunto (0.75 vs 0.61), indicando mayor equilibrio entre no inventar citas y aprovechar parte del contexto.
* LLM‑only es ligeramente inferior en promedio para el juez y, crucialmente, no ofrece trazabilidad (no aplica GCF1).

---

### *2. Resultados con "repair"*

**Hybrid‑RAG (repaired):**

* Judge: score_total = 4.0 (↓)
* Groundedness: GCF1 = 0.80 (↑ por mayor recall)
* Leakage: 1.0 (emerge fuga/penalización)



**Dense‑RAG (repaired):**

* Judge: score_total = 4.0 (↓)
* Groundedness: GCF1 ≈ 0.29 (↓)
* Leakage: 2.0 (↑)



**Interpretación clave:**

El repair pass no garantiza mejoría global. Aunque puede subir GCF1 (aprovechamiento del contexto) en ciertos casos, degrada la calidad percibida por el juez (naturalidad/tono/fidelidad narrativa) y puede aumentar la detección de inconsistencias (leakage según la política estricta). En síntesis, no es recomendable como paso automático por defecto.

---

### *3. Estabilidad (desviación estándar)*

**Dense y Hybrid** (sin repair) muestran **baja variabilidad** en las métricas del juez y GCF1 en este conjunto, lo que sugiere consistencia entre consultas.
**LLM‑only** mantiene una **mayor variabilidad** en dimensiones semánticas, consistente con el carácter generativo sin anclaje a evidencias.

---

### *4. Impacto del RAG y robustez comparada*

**RAG** aporta trazabilidad y consistencia frente a LLM‑only.
En este conjunto, Dense‑RAG y Hybrid‑RAG son prácticamente equivalentes en evaluación del juez; con GCF1, **Dense‑RAG** obtiene una ligera ventaja al exhibir mejor balance entre precisión y aprovechamiento del contexto.
**LLM‑only** es más rápido, pero carece de garantías de groundedness (no aplican métricas citacionales).

---

### *5. Implicación para LUMINA*

Para un sistema pastoral verificable y responsable:

* **Dense‑RAG es suficiente y robusto**; ofrece alto score del juez, cero fuga, y mejor GCF1 en promedio que Hybrid en este conjunto.
* Hybrid‑RAG no muestra ventaja significativa en generación bajo estas condiciones; su GCF1 es competitivo pero inferior.
* El repair automático no debe usarse por defecto: conviene reservarlo para casos detectados de fuga o baja groundedness y ejecutarlo de forma condicionada.

---

### **Conclusión Final del Avance 4**

* El baseline es sólido: Dense‑RAG y Hybrid‑RAG alcanzan desempeño máximo en el juez con cero fuga.

* GCF1 agrega valor decisivo: permite distinguir no sólo si la respuesta es buena, sino qué tan bien usa el contexto. Bajo GCF1, Dense‑RAG muestra una ligera superioridad en groundedness frente a Hybrid, manteniendo precisión perfecta y aceptable aprovechamiento del contexto.
* LLM‑only sigue siendo una línea base fuerte en calidad lingüística y latencia, pero no es trazable y no ofrece garantías de fidelidad a evidencias.
* El repair pass no mejora la calidad global de forma consistente y puede deteriorarla; su uso debe ser selectivo.

---

> *“Pregunta ahora a las bestias, y ellas te enseñarán; a las aves de los cielos, y ellas te lo mostrarán… ¿Quién no sabe que la mano de Jehová ha hecho esto?”*
>
> Job 12:7-9

---